# LEBIS — อัลกอริทึมของระบบยืม-คืนอุปกรณ์ (ฉบับรวม)

**วิชา Data Structures & Algorithms**

| ส่วน | คนเขียน | อัลกอริทึมที่เขียนเอง |
|---|---|---|
| Insert / Delete | Kim | Hash Table (chaining + resize) |
| Search / Count | Neo | Binary Search |
| Sort / Group | Kul | Merge Sort |

ทุกคนวัดบนชุดข้อมูล A–D ชุดเดียวกัน · **วิธีรัน:** Runtime → Run all

## 1. ไฟล์ของทั้งทีม

แต่ละเซลล์คือไฟล์จริงจากรีโป เขียนลงเป็นไฟล์แล้ว import แยกกัน ชื่อฟังก์ชันที่ซ้ำกันข้ามไฟล์จึงไม่ทับกัน

In [1]:
%%writefile datasets.py
"""ชุดข้อมูลทดสอบ A–D — งานส่งวิชา Data Structures & Algorithms ของ Neo

ทั้งสามหัวข้อของกลุ่ม (hash_table ของ Kim · search ของ Neo · sort_group ของ Kul)
ต้องวัดผลบนข้อมูลชุดเดียวกันจากไฟล์นี้ ไม่ใช่ต่างคนต่างสร้างข้อมูลของตัวเอง
ถ้าแต่ละคนวัดบนข้อมูลคนละชุด ตัวเลขในรายงานจะอ่านเทียบกันไม่ได้เลย
และกรรมการจะเห็นทันทีว่าสามหัวข้อไม่ได้คุยกัน

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

วิธีรัน
    python3 backend-api/algorithms/datasets.py

วิธีใช้ใน Colab
    วางเซลล์ของไฟล์นี้ไว้ "ก่อน" เซลล์ของ hash_table.py และ search.py
    แล้วเรียก make_dataset("A") ... make_dataset("D") ส่งเข้าฟังก์ชันของแต่ละคน
    ได้ตรงๆ เพราะทุกตัวรับ list ของ dict ธรรมดา

หนึ่ง record มี 8 ฟิลด์ตามที่ Kim ออกแบบไว้
    code       รหัสอุปกรณ์ ไม่ซ้ำกันทั้งชุด — ใช้เป็น key ของตารางแฮช
    name       ชื่อที่คนเรียก ซ้ำกันได้ (ชุด B จงใจให้ซ้ำ)
    category   หมวด ใช้ตอนจัดกลุ่มของ Kul
    total      จำนวนทั้งหมดที่ร้านมี
    available  จำนวนที่ยังอยู่บนชั้นตอนนี้
    borrower   ชื่อผู้เช่า ถ้ายังไม่มีใครยืมเป็น None
    due_date   กำหนดคืนในรูป "YYYY-MM-DD" ถ้าไม่มีคนยืมเป็น None
    status     "ok" ว่างให้เช่าได้ · "out" ถูกยืมหมด · "broken" ชำรุด

ทำไมวันที่ถึงตายตัว
    count_overdue() ของ search.py นับจากวันครบกำหนดเทียบกับ "วันนี้"
    ถ้าใช้วันจริงของเครื่อง ตัวเลขในรายงานจะเปลี่ยนทุกวันที่รันใหม่
    และตัวเลขในรายงานกับตัวเลขในวิดีโอจะไม่ตรงกัน ไฟล์นี้จึงตรึง TODAY ไว้
    ให้ทุกคนอ้างอิงค่าเดียวกัน
"""

import random
from datetime import date, timedelta

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — ค่าคงที่
# ---------------------------------------------------------------------------

# วันอ้างอิงตายตัว ใช้แทน "วันนี้" ทั้งระบบทดสอบ
# ตัวเลขในรายงานจึงเท่าเดิมทุกครั้งที่รัน ไม่ว่าจะรันวันไหน
TODAY = date(2026, 10, 1)

# หมวดและคำนำหน้ารหัส ใช้ชุดเดียวกับ hash_table.py ของ Kim
# (คัดลอกค่ามา ไม่ได้ import เพราะกฎ stdlib เท่านั้นข้างบน)
CATEGORIES = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]

CATEGORY_PREFIX = {
    "กล้อง": "CAM",
    "เลนส์": "LEN",
    "ขาตั้ง": "TRI",
    "ไฟ": "LED",
    "ไมโครโฟน": "MIC",
}

# ชื่อผู้เช่าสำหรับข้อมูลจำลอง ไม่ใช่ชื่อคนจริง
BORROWERS = [
    "ปวีณา",
    "ธนกร",
    "สุชาดา",
    "ณัฐพล",
    "กมลชนก",
    "อานนท์",
    "พิมพ์ชนก",
    "วีรภัทร",
]

# seed แยกตามชุด ชุดเดิมจึงได้ข้อมูลเดิมเสมอ แต่คนละชุดไม่เหมือนกัน
SEEDS = {"A": 11, "B": 22, "C": 33, "D": 44}

DATASET_SIZES = {"A": 10, "B": 12, "C": 10, "D": 1000}


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — ตัวช่วยสร้าง record
# ---------------------------------------------------------------------------


def _iso(day: date) -> str:
    """แปลงวันเป็นข้อความ "YYYY-MM-DD"

    เก็บเป็นข้อความไม่ใช่ออบเจ็กต์ date เพราะรูปแบบนี้เรียงลำดับด้วยการเทียบ
    ข้อความได้ตรงๆ อยู่แล้ว (ปี-เดือน-วัน เรียงจากหน่วยใหญ่ไปเล็ก)
    merge_sort ของ Kul กับ binary_search ของ Neo จึงเทียบวันได้โดยไม่ต้องแปลงก่อน
    """
    return day.isoformat()


def _make_record(
    number: int,
    category: str,
    name: str,
    total: int,
    available: int,
    borrower=None,
    due_offset=None,
    broken: bool = False,
) -> dict:
    """ประกอบ record 8 ฟิลด์ให้ครบ และคิดค่า status ให้สอดคล้องกับ available

    due_offset คือจำนวนวันนับจาก TODAY ติดลบ = เลยกำหนดคืนมาแล้ว
    """
    if broken:
        status = "broken"
    elif available == 0:
        status = "out"
    else:
        status = "ok"

    return {
        "code": f"{CATEGORY_PREFIX[category]}-{number:04d}",
        "name": name,
        "category": category,
        "total": total,
        "available": available,
        "borrower": borrower,
        "due_date": _iso(TODAY + timedelta(days=due_offset)) if due_offset is not None else None,
        "status": status,
    }


def _build(size: int, seed: int, duplicate_names: bool, zero_stock: bool) -> list:
    """สร้างรายการตามพารามิเตอร์ที่แต่ละชุดต้องการ

    duplicate_names  ให้ชื่อซ้ำกันเป็นคู่ๆ แต่ code ยังไม่ซ้ำ (ชุด B)
    zero_stock       ดันให้หลายตัวมี available = 0 (ชุด C)
    """
    generator = random.Random(seed)
    items = []

    for number in range(1, size + 1):
        # ชุด B: ตัวเลขคู่ใช้ชื่อเดียวกับตัวก่อนหน้า จึงได้ชื่อซ้ำเป็นคู่ๆ
        # แต่ code ยังไม่ซ้ำ เพราะ code เดินตาม number ตลอด
        # ใช้พิสูจน์ว่าค้นด้วยชื่อแล้วยังแยกออกว่าเป็นตัวไหน
        #
        # หมวดต้องเดินตามชื่อไปด้วย ไม่งั้นจะได้ของที่ชื่อว่า "เลนส์ รุ่นที่ 1"
        # แต่หมวดเป็น "ขาตั้ง" ซึ่งเป็นข้อมูลที่ขัดกันเองและทำให้ group_by_type
        # ของ Kul ให้ผลที่อธิบายในรายงานไม่ได้
        name_number = number - 1 if duplicate_names and number % 2 == 0 else number
        category = CATEGORIES[name_number % len(CATEGORIES)]
        name = f"{category} รุ่นที่ {name_number}"

        total = generator.randint(1, 5)

        if zero_stock and number % 2 == 1:
            # ชุด C: ครึ่งหนึ่งถูกยืมไปหมด
            available = 0
        else:
            available = generator.randint(0, total)

        borrowed_units = total - available

        if borrowed_units > 0:
            borrower = BORROWERS[number % len(BORROWERS)]
            # กระจายกำหนดคืนไว้ทั้งก่อนและหลังวันอ้างอิง
            # ค่าติดลบคือเลยกำหนดแล้ว ซึ่งเป็นตัวที่ count_overdue() ต้องนับได้
            due_offset = generator.randint(-10, 14)
        else:
            borrower = None
            due_offset = None

        # ให้มีของชำรุดปนอยู่บ้าง เพื่อให้ count_by_status() มีครบทั้งสามสถานะ
        # ทุกๆ 7 ชิ้น — เลือกเลขนี้เพราะชุดเล็กสุดมี 10 ชิ้น ใช้ 7 แล้วยังได้
        # อย่างน้อย 1 ชิ้นในทุกชุด ถ้าใช้เลขที่ใหญ่กว่านั้น ชุด A-C จะไม่มีของชำรุดเลย
        # แล้วการนับสถานะ "broken" จะไม่เคยถูกทดสอบบนชุดที่ตรวจด้วยตาได้
        broken = number % 7 == 0

        items.append(
            _make_record(
                number=number,
                category=category,
                name=name,
                total=total,
                available=available,
                borrower=borrower,
                due_offset=due_offset,
                broken=broken,
            )
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — ทางเข้าหลัก
# ---------------------------------------------------------------------------


def make_dataset(name: str) -> list:
    """คืนชุดข้อมูลทดสอบตามชื่อชุด "A" "B" "C" หรือ "D"

    A  10 รายการ  ข้อมูลสะอาด ชื่อไม่ซ้ำ       ใช้เช็กว่าทุกฟังก์ชันทำงานถูก
    B  12 รายการ  ชื่อซ้ำเป็นคู่ code ไม่ซ้ำ    ใช้เช็กว่าค้นแล้วได้ตัวที่ถูก
    C  10 รายการ  หลายตัว available = 0        ใช้เช็กการนับและสถานะ
    D  1,000 รายการ                            ใช้วัดเวลาจริงเทียบ Big-O

    เรียกซ้ำกี่ครั้งก็ได้ข้อมูลเดิมเสมอ เพราะ seed ตายตัวตาม SEEDS
    แต่คืน list ใหม่ทุกครั้ง ฟังก์ชันที่แก้ข้อมูล (เช่น sort แบบแก้ในที่)
    จึงไม่ไปกระทบผลของคนอื่นที่เรียกทีหลัง
    """
    key = name.strip().upper()

    if key not in DATASET_SIZES:
        raise ValueError(f"ไม่รู้จักชุดข้อมูล {name!r} — มีให้เลือกแค่ A, B, C, D")

    return _build(
        size=DATASET_SIZES[key],
        seed=SEEDS[key],
        duplicate_names=(key == "B"),
        zero_stock=(key == "C"),
    )


def all_datasets() -> dict:
    """คืนทั้งสี่ชุดในทีเดียว ใช้ตอนวนวัดผลในรายงาน"""
    return {name: make_dataset(name) for name in ("A", "B", "C", "D")}


def describe(items: list) -> dict:
    """สรุปหน้าตาของชุดข้อมูลไว้ตรวจว่าสร้างมาถูกจริง

    ไม่ได้เรียกฟังก์ชันนับของ search.py เพราะไฟล์นี้ต้องรันได้ลำพัง
    (และ search.py ก็ต้องรันได้ลำพังเหมือนกัน) การนับสองบรรทัดตรงนี้
    จึงถูกกว่าการผูกสองไฟล์เข้าหากัน
    """
    codes = [item["code"] for item in items]
    names = [item["name"] for item in items]

    return {
        "จำนวน": len(items),
        "code ไม่ซ้ำ": len(set(codes)) == len(codes),
        "ชื่อที่ซ้ำกัน": len(names) - len(set(names)),
        "available = 0": sum(1 for item in items if item["available"] == 0),
        "มีคนยืมอยู่": sum(1 for item in items if item["borrower"] is not None),
        "เลยกำหนดคืน": sum(
            1
            for item in items
            if item["due_date"] is not None and item["due_date"] < _iso(TODAY)
        ),
        "ชำรุด": sum(1 for item in items if item["status"] == "broken"),
    }


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — ตรวจว่าแต่ละชุดมีคุณสมบัติตามที่ตั้งใจไว้จริง
#
# ข้อมูลทดสอบที่ผิดเงียบๆ อันตรายกว่าไม่มีข้อมูลทดสอบ เพราะทุกฟังก์ชัน
# จะ "ผ่าน" หมดแล้วเราจะเชื่อผลนั้นไปเขียนในรายงาน
# ---------------------------------------------------------------------------


def verify() -> None:
    """เช็กคุณสมบัติที่แต่ละชุดสัญญาไว้ ถ้าไม่ตรงให้ล้มทันที"""
    a = make_dataset("A")
    b = make_dataset("B")
    c = make_dataset("C")
    d = make_dataset("D")

    # ขนาดตามที่แผนกำหนด — โจทย์บังคับว่าทุกชุดต้อง >= 10
    assert len(a) == 10 and len(b) == 12 and len(c) == 10 and len(d) == 1000
    assert all(len(items) >= 10 for items in (a, b, c, d))

    # code ต้องไม่ซ้ำทุกชุด เพราะเป็น key ของตารางแฮช
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        codes = [item["code"] for item in items]
        assert len(set(codes)) == len(codes), f"ชุด {name} มี code ซ้ำ"

    # ชุด A ชื่อต้องไม่ซ้ำ ชุด B ต้องซ้ำ — นี่คือความต่างที่ชุด B มีไว้พิสูจน์
    assert len({item["name"] for item in a}) == len(a), "ชุด A ไม่ควรมีชื่อซ้ำ"
    assert len({item["name"] for item in b}) < len(b), "ชุด B ต้องมีชื่อซ้ำ"

    # ชุด C ต้องมีของหมดสต็อกอย่างน้อยครึ่งหนึ่ง
    assert sum(1 for item in c if item["available"] == 0) >= len(c) // 2

    # ทุกชุดต้องมีของชำรุดอย่างน้อยหนึ่งชิ้น ไม่งั้น count_by_status()
    # จะไม่เคยถูกทดสอบครบทั้งสามสถานะ
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        assert any(item["status"] == "broken" for item in items), f"ชุด {name} ไม่มีของชำรุด"

    # ทุกชุดต้องมีของที่เลยกำหนดคืน ไม่งั้น count_overdue() จะทดสอบไม่ได้
    for name, items in (("A", a), ("B", b), ("C", c), ("D", d)):
        overdue = [
            item
            for item in items
            if item["due_date"] is not None and item["due_date"] < _iso(TODAY)
        ]
        assert overdue, f"ชุด {name} ไม่มีรายการที่เลยกำหนดคืนเลย"

    # ความสอดคล้องของ record ทุกใบ
    for items in (a, b, c, d):
        for item in items:
            assert set(item) == {
                "code",
                "name",
                "category",
                "total",
                "available",
                "borrower",
                "due_date",
                "status",
            }, "record ต้องมีครบ 8 ฟิลด์"
            assert 0 <= item["available"] <= item["total"], "ของว่างต้องไม่เกินของทั้งหมด"
            assert item["status"] in ("ok", "out", "broken")

            # มีคนยืมก็ต้องมีกำหนดคืน ไม่มีคนยืมก็ต้องไม่มีกำหนดคืน
            assert (item["borrower"] is None) == (item["due_date"] is None)

            # ของที่ยังไม่มีใครยืมต้องมีของครบบนชั้น
            if item["borrower"] is None:
                assert item["available"] == item["total"]

    # เรียกสองครั้งต้องได้ข้อมูลเดิมเป๊ะ ไม่งั้นตัวเลขในรายงานจะขยับเอง
    assert make_dataset("D") == d, "make_dataset ต้องคืนข้อมูลเดิมทุกครั้ง"

    # แต่ต้องเป็น list คนละใบ ไม่ใช่ใบเดิมที่ใครแก้ไปแล้ว
    assert make_dataset("A") is not a

    print("ตรวจชุดข้อมูล A-D ผ่านทั้งหมด")


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — รันเพื่อดูผล
# ---------------------------------------------------------------------------


def main() -> None:
    print("=" * 70)
    print("ชุดข้อมูลทดสอบ A-D ของ LEBIS")
    print(f"วันอ้างอิง (TODAY) = {_iso(TODAY)}")
    print("=" * 70)

    for name in ("A", "B", "C", "D"):
        items = make_dataset(name)
        print()
        print(f"ชุด {name}")
        print("-" * 70)
        for label, value in describe(items).items():
            print(f"    {label:<16} {value}")

    print()
    print("ตัวอย่าง 3 รายการแรกของชุด A")
    print("-" * 70)
    for item in make_dataset("A")[:3]:
        print(f"    {item}")

    print()
    print("ชื่อที่ซ้ำกันในชุด B (code ไม่ซ้ำ)")
    print("-" * 70)
    for item in make_dataset("B"):
        print(f"    {item['code']}  {item['name']}")

    print()
    verify()


if __name__ == "__main__":
    main()


Writing datasets.py


In [2]:
%%writefile hash_table.py
"""ตารางแฮชเขียนเอง (hash table) — งานส่งวิชา Data Structures & Algorithms ของ Kim

ใช้ในระบบ LEBIS เพื่อค้นอุปกรณ์จากรหัส (code) ให้เร็วขึ้น
จากเดิมที่ต้องวนดูทั้งรายการทีละชิ้น

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

วิธีรัน
    python3 backend-api/algorithms/hash_table.py

สิ่งที่ต้องรู้ก่อนอ่านโค้ด
    ตารางแฮชคือการ "คำนวณ" ว่าข้อมูลควรอยู่ช่องไหน แทนการ "ไล่หา" ทีละช่อง
    พอรู้ช่องได้จากการคำนวณ เวลาค้นจึงไม่ขึ้นกับว่ามีข้อมูลทั้งหมดกี่ชิ้น
"""

import random
import time

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — ตารางแฮช
# ---------------------------------------------------------------------------


class HashTable:
    """ตารางแฮชที่เขียนเองทั้งหมด แก้การชนด้วยการต่อโซ่ (chaining)

    ที่เก็บข้อมูลจริงคือ list ของ list ชื่อ self._buckets
    ไม่ได้ใช้ dict ของ Python เป็นที่เก็บ (ใช้ dict ก็คือให้ Python ทำงานให้
    ซึ่งไม่ใช่โจทย์ของวิชานี้) โครงหน้าตาประมาณนี้

        self._buckets = [
            [],                                    <- ถังที่ 0 ว่าง
            [("CAM-01", {...}), ("LEN-07", {...})], <- ถังที่ 1 มี 2 ตัว = ชนกัน
            [("TRI-03", {...})],                   <- ถังที่ 2 มี 1 ตัว
            ...
        ]

    แต่ละถัง (bucket) เก็บคู่ (key, value) เป็น tuple ต่อกันเป็นโซ่
    """

    # ค่าเริ่มต้นของจำนวนถัง ใช้เลขยกกำลังสองเพื่อให้เห็นการขยายชัดๆ (8 -> 16 -> 32)
    INITIAL_CAPACITY = 8

    # เกินค่านี้แล้วขยายตาราง — ที่มาของเลข 0.7 อธิบายไว้ที่ _resize()
    MAX_LOAD_FACTOR = 0.7

    # ฐานของ polynomial rolling hash (เหตุผลที่เลือก 31 อยู่ใน _hash())
    HASH_BASE = 31

    # จำนวนเฉพาะขนาดใหญ่ ใช้มอดเพื่อไม่ให้เลขโตจนคำนวณช้า
    HASH_MODULUS = 1_000_000_007

    def __init__(self, capacity: int = INITIAL_CAPACITY):
        # ที่เก็บข้อมูลจริง: list ของ list
        self._buckets = [[] for _ in range(capacity)]

        # จำนวนคู่ข้อมูลที่เก็บอยู่จริง (ไม่ใช่จำนวนถัง)
        self._size = 0

        # ตัวนับไว้เขียนรายงาน — นับเฉพาะตอน insert ของผู้ใช้
        # ไม่นับตอน rehash ภายใน ไม่งั้นตัวเลขจะพุ่งเกินความจริง
        self.collisions = 0
        self.resize_count = 0

    # -- ฟังก์ชันแฮช ------------------------------------------------------

    def _hash(self, key) -> int:
        """แปลง key เป็นเลขถัง ด้วยวิธี polynomial rolling hash

        สูตร: h = (((c1 * 31 + c2) * 31 + c3) * 31 + ...) % ตัวมอด
        แล้วค่อยเอา % จำนวนถัง เพื่อให้ได้เลขที่อยู่ในช่วงถังที่มีอยู่จริง

        ทำไมเลือกวิธีนี้ (ไม่ใช่แค่บวก ord ของทุกตัวอักษรรวมกัน)
          1. ตำแหน่งของตัวอักษรมีผลต่อผลลัพธ์ — "CAM-01" กับ "10-MAC" ได้ค่าต่างกัน
             ถ้าใช้วิธีบวกเฉยๆ คำที่สลับตัวอักษรกันจะได้ค่าเท่ากันหมด
             แล้วรหัสอุปกรณ์ของร้านเช่ามักต่างกันแค่การสลับตัวเลขท้าย
             (CAM-01 · CAM-10) ซึ่งจะชนกันรัวๆ
          2. คำนวณเร็ว วนตัวอักษรรอบเดียว = O(ความยาวของ key)
             ไม่ต้องยกกำลังจริง เพราะคูณฐานสะสมไปเรื่อยๆ (นี่คือความหมายของ rolling)
          3. ฐาน 31 เป็นจำนวนเฉพาะที่เล็กพอให้คูณเร็ว และใหญ่พอให้ค่ากระจาย
             เป็นค่าที่ไลบรารีจริงหลายตัวใช้ (เช่น String.hashCode ของ Java)
          4. มอดด้วยจำนวนเฉพาะใหญ่ (1e9+7) กัน int โตไม่จำกัด
             Python คำนวณเลขใหญ่ได้ แต่ยิ่งใหญ่ยิ่งช้า
        """
        # คิดบนรูปข้อความของ key — ในระบบ LEBIS key คือรหัสอุปกรณ์ซึ่งเป็นข้อความอยู่แล้ว
        #
        # ข้อจำกัดที่ต้องรู้: ตารางนี้ "เทียบ" key ด้วย == แต่ "แฮช" จากข้อความ
        # ปกติสองอย่างนี้ต้องสอดคล้องกัน คือ key ที่เท่ากันต้องแฮชลงถังเดียวกัน
        # ซึ่งจริงเสมอสำหรับข้อความ แต่ไม่จริงกับบางชนิดของ Python เช่น
        # 1 == True เป็นจริง แต่ str(1) = "1" กับ str(True) = "True" ต่างกัน
        # ตารางจะมองเป็นสอง key แยกกัน (dict ของ Python มองเป็น key เดียว)
        # จึงควรใช้ key ชนิดเดียวกันทั้งตาราง — ของเราใช้ข้อความล้วน
        text = str(key)

        hash_value = 0
        for character in text:
            hash_value = (hash_value * self.HASH_BASE + ord(character)) % self.HASH_MODULUS

        return hash_value % len(self._buckets)

    # -- เพิ่ม / อ่าน / ลบ -------------------------------------------------

    def insert(self, key, value) -> None:
        """ใส่คู่ (key, value) — key เดิมซ้ำถือว่าแก้ค่าทับ ไม่ใช่เพิ่มตัวใหม่

        เรื่อง "การชน" (collision)
            การชนคือสถานการณ์ที่ key ต่างกัน แต่ _hash() คำนวณได้เลขถังเดียวกัน
            เป็นเรื่องที่เลี่ยงไม่ได้ เพราะ key มีได้ไม่จำกัดแบบ แต่ถังมีจำกัด
            (ต่อให้ฟังก์ชันแฮชดีแค่ไหน ใส่ของ 9 ชิ้นลงตาราง 8 ถัง ก็ต้องมีถัง
            ที่ได้ของ 2 ชิ้นอยู่แล้ว)

            วิธีแก้ในไฟล์นี้คือ chaining — ต่อโซ่: ถังหนึ่งเก็บได้หลายคู่
            โดยเก็บเป็น list ต่อกันไป ตอนค้นก็ไปที่ถังนั้นแล้วไล่ดูในโซ่สั้นๆ
            (อีกวิธีที่นิยมคือ open addressing ที่เลื่อนไปหาถังว่างถัดไป
            แต่ chaining อ่านง่ายกว่าและลบข้อมูลตรงไปตรงมากว่า)
        """
        index = self._hash(key)
        bucket = self._buckets[index]

        # key เดิมมีอยู่แล้วหรือยัง — ถ้ามีก็แก้ค่าทับ จำนวนข้อมูลไม่เพิ่ม
        for position, (existing_key, _existing_value) in enumerate(bucket):
            if existing_key == key:
                bucket[position] = (key, value)
                return

        # มาถึงบรรทัดนี้แปลว่าเป็น key ใหม่
        # ถ้าถังไม่ว่างอยู่แล้ว = ชน (นับไว้เขียนรายงาน)
        if len(bucket) > 0:
            self.collisions += 1

        bucket.append((key, value))
        self._size += 1

        # ใส่เสร็จแล้วค่อยเช็กว่าตารางแน่นเกินไปมั้ย
        if self.load_factor() > self.MAX_LOAD_FACTOR:
            self._resize()

    def get(self, key):
        """คืน value ของ key นั้น ถ้าไม่มีคืน None

        ไปที่ถังที่คำนวณได้ทันที (ไม่ต้องไล่ทุกถัง) แล้วไล่ดูในโซ่ของถังนั้น
        """
        bucket = self._buckets[self._hash(key)]

        for existing_key, existing_value in bucket:
            if existing_key == key:
                return existing_value

        return None

    def delete(self, key) -> bool:
        """ลบ key ออก คืน True ถ้าลบสำเร็จ คืน False ถ้าไม่มี key นั้นอยู่ตั้งแต่แรก"""
        bucket = self._buckets[self._hash(key)]

        for position, (existing_key, _existing_value) in enumerate(bucket):
            if existing_key == key:
                bucket.pop(position)
                self._size -= 1
                return True

        return False

    # -- การขยายตาราง ------------------------------------------------------

    def load_factor(self) -> float:
        """ความแน่นของตาราง = จำนวนข้อมูล ÷ จำนวนถัง

        ค่านี้คือความยาวเฉลี่ยของโซ่ในแต่ละถัง ซึ่งคือจำนวนครั้งที่ต้องเทียบ
        โดยเฉลี่ยตอนค้นหา — เลยเป็นตัวเลขที่บอกว่าตารางยัง "เร็ว" อยู่มั้ย
        """
        return self._size / len(self._buckets)

    def _resize(self) -> None:
        """ขยายจำนวนถังเป็น 2 เท่า แล้วคำนวณตำแหน่งใหม่ให้ข้อมูลทุกตัว (rehash)

        ทำไมต้องขยายตอน load factor เกิน 0.7
            load factor คือความยาวเฉลี่ยของโซ่ ปล่อยให้แน่นขึ้นเรื่อยๆ โซ่จะยาวขึ้น
            การค้นก็ค่อยๆ กลายเป็นการไล่ดูทีละตัวเหมือน list ธรรมดา
            0.7 เป็นค่าที่ใช้กันทั่วไปเพราะอยู่ตรงกลางระหว่างสองอย่างที่ขัดกัน
              - ตั้งต่ำเกิน (เช่น 0.2) ตารางโปร่งแต่กินหน่วยความจำเยอะ และขยายบ่อย
              - ตั้งสูงเกิน (เช่น 0.95) ประหยัดที่แต่โซ่ยาว ค้นช้า

        ทำไมต้องคำนวณตำแหน่งใหม่ทุกตัว
            เลขถังมาจาก hash % จำนวนถัง — พอจำนวนถังเปลี่ยน ผลของ % ก็เปลี่ยน
            ถ้าย้ายของเฉยๆ โดยไม่คำนวณใหม่ จะหาของไม่เจออีกเลย

        การขยายมีต้นทุน O(n) หนึ่งครั้ง แต่เกิดนานๆ ครั้ง (ทุกครั้งที่ข้อมูล
        เพิ่มเป็นเท่าตัว) เฉลี่ยต่อการ insert หนึ่งครั้งจึงยังถือเป็น O(1)
        ศัพท์ของเรื่องนี้เรียกว่า amortized O(1)
        """
        old_buckets = self._buckets

        self._buckets = [[] for _ in range(len(old_buckets) * 2)]
        self.resize_count += 1

        # ใส่ของกลับเข้าตารางใหม่ — ไม่แตะตัวนับ collisions เพราะไม่ใช่การชน
        # ที่เกิดจากการใช้งานของผู้ใช้ แต่เป็นงานบ้านภายในของตาราง
        for bucket in old_buckets:
            for key, value in bucket:
                self._buckets[self._hash(key)].append((key, value))

    # -- ข้อมูลทั่วไปของตาราง ---------------------------------------------

    def __len__(self) -> int:
        """ให้ len(table) ใช้ได้ — คืนจำนวนคู่ข้อมูล ไม่ใช่จำนวนถัง"""
        return self._size

    def keys(self) -> list:
        """คืน key ทั้งหมดเป็น list

        ลำดับที่ได้เป็นลำดับของถัง ไม่ใช่ลำดับที่ใส่เข้ามา — ตารางแฮช
        ไม่รับประกันลำดับ ถ้าต้องการเรียงให้เอาไปเรียงเองข้างนอก
        """
        collected = []
        for bucket in self._buckets:
            for key, _value in bucket:
                collected.append(key)
        return collected

    def stats(self) -> dict:
        """ตัวเลขสถานะของตาราง ไว้ใส่รายงาน"""
        chain_lengths = [len(bucket) for bucket in self._buckets]
        used_buckets = sum(1 for length in chain_lengths if length > 0)

        return {
            "จำนวนข้อมูล": self._size,
            "จำนวนถัง": len(self._buckets),
            "ถังที่ถูกใช้": used_buckets,
            "load factor": round(self.load_factor(), 3),
            "ชนทั้งหมด": self.collisions,
            "ขยายตาราง": self.resize_count,
            "โซ่ที่ยาวสุด": max(chain_lengths) if chain_lengths else 0,
        }


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — ความเร็ว: ทำไมบอกว่า O(1) เป็นค่าเฉลี่ย ไม่ใช่กรณีแย่สุด
#
# ค้นหาในตารางแฮช
#   กรณีเฉลี่ย O(1) — คำนวณเลขถังครั้งเดียว (ไม่ขึ้นกับ n) แล้วไล่โซ่ในถังนั้น
#                     ความยาวโซ่เฉลี่ย = load factor ซึ่งถูกกดไว้ไม่เกิน 0.7
#                     ด้วยการขยายตาราง เลยถือเป็นค่าคงที่ ไม่โตตาม n
#   กรณีแย่สุด O(n) — ถ้าทุก key ไปกองอยู่ถังเดียวกัน ตารางจะเหลือโซ่เส้นเดียว
#                     ยาว n ตัว การค้นก็กลายเป็นการไล่ดูทีละตัวเหมือน list
#
# กรณีแย่สุดเกิดขึ้นเมื่อไหร่จริงๆ
#   1. ฟังก์ชันแฮชแย่ เช่นเขียน return 0 หรือใช้ len(key) เป็นเลขถัง
#      (รหัสอุปกรณ์ยาวเท่ากันหมดก็ได้ถังเดียวกันหมด)
#   2. ไม่ขยายตาราง — ข้อมูลเพิ่มแต่ถังเท่าเดิม โซ่ยาวขึ้นเรื่อยๆ
#   3. ข้อมูลที่จงใจเลือกมาให้ชน (algorithmic complexity attack) คนที่รู้สูตรแฮช
#      ของเราสามารถคิด key ที่แฮชลงถังเดียวกันทั้งหมดได้ ของจริงจึงผสมค่าสุ่ม
#      (salt/seed) ลงในฟังก์ชันแฮช — Python เองก็ทำ ดู PYTHONHASHSEED
#
# สรุปคือ O(1) ไม่ได้มาฟรี มันมาจากฟังก์ชันแฮชที่กระจายดี + การขยายตาราง
# ขาดข้อใดข้อหนึ่งก็ไหลลงไปเป็น O(n)
# ---------------------------------------------------------------------------


def find_with_list_scan(items: list, code: str):
    """วิธีที่ 1 — ไล่ดูทีละตัวใน list (linear search) O(n)"""
    for item in items:
        if item["code"] == code:
            return item
    return None


def delete_with_list_scan(items: list, code: str) -> bool:
    """ลบออกจาก list — ต้องไล่หาตำแหน่งก่อน แล้วค่อยลบ O(n)"""
    for position, item in enumerate(items):
        if item["code"] == code:
            items.pop(position)
            return True
    return False


def build_hash_table(items: list) -> HashTable:
    """ยกข้อมูลทั้งชุดใส่ตารางแฮชที่เราเขียน โดยใช้ code เป็น key"""
    table = HashTable()
    for item in items:
        table.insert(item["code"], item)
    return table


def build_python_dict(items: list) -> dict:
    """วิธีที่ 3 — dict ของ Python ไว้เป็นตัวเทียบว่าของเราห่างจากของจริงแค่ไหน"""
    return {item["code"]: item for item in items}


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — เครื่องมือวัดเวลา
# ---------------------------------------------------------------------------

ROUNDS = 3  # รันซ้ำกี่รอบแล้วเฉลี่ย — กันจังหวะที่เครื่องไปทำอย่างอื่นชั่วคราว


def measure(operation, setup=None, rounds: int = ROUNDS) -> float:
    """วัดเวลาเฉลี่ยของ operation หน่วยเป็นมิลลิวินาที

    ใช้ time.perf_counter() ไม่ใช่ time.time() เพราะ perf_counter เป็นนาฬิกา
    ความละเอียดสูงที่ทำมาเพื่อวัดช่วงเวลาสั้นๆ และไม่กระโดดเวลาเครื่องปรับนาฬิกา

    setup คือฟังก์ชันที่เตรียมของให้ใหม่ก่อนทุกรอบ (เช่นสร้างตารางขึ้นมาใหม่
    ก่อนวัดการลบ) จุดสำคัญคือมันถูกเรียก "ก่อน" เริ่มจับเวลา เวลาที่ใช้สร้างของ
    จึงไม่ถูกนับรวมเข้าไปในผล ถ้านับรวม เราจะวัดเวลาสร้าง+ลบ ไม่ใช่เวลาลบ
    """
    durations = []

    for _ in range(rounds):
        target = setup() if setup is not None else None

        started_at = time.perf_counter()
        operation(target)
        durations.append(time.perf_counter() - started_at)

    return sum(durations) / len(durations) * 1000


def lookup_order(items: list, seed: int = 99) -> list:
    """คืนลำดับ code ที่สลับแล้ว ไว้ใช้เป็นลำดับในการค้นและลบ

    ทำไมต้องสลับ: ถ้าไล่ตามลำดับเดิม การลบออกจาก list จะกลายเป็นการลบตัวแรก
    ทุกครั้ง ซึ่งเป็น "กรณีดีที่สุด" ของ list (เจอที่ตำแหน่ง 0 ทันที)
    ตัวเลขที่ได้จะสวยเกินความจริงและสรุปผิดว่า list ลบเร็วกว่าตารางแฮช
    การสลับลำดับทำให้ตำแหน่งที่ต้องไล่หาเฉลี่ยอยู่กลาง list = กรณีทั่วไปจริงๆ

    ใส่ seed ไว้ให้ได้ลำดับเดิมทุกครั้ง ตัวเลขในรายงานจะเทียบกันได้
    """
    codes = [item["code"] for item in items]
    random.Random(seed).shuffle(codes)
    return codes


def benchmark_lookup(items: list) -> dict:
    """วัดเวลา "ค้นหาทุก key หนึ่งครั้ง" ของทั้งสามวิธี บนข้อมูลชุดเดียวกัน

    การค้นไม่ได้แก้ข้อมูล จึงสร้างของไว้ล่วงหน้าครั้งเดียว ใช้ซ้ำได้ทุกรอบ
    """
    codes = lookup_order(items)

    table = build_hash_table(items)
    python_dict = build_python_dict(items)

    def with_list(_unused):
        for code in codes:
            find_with_list_scan(items, code)

    def with_hash_table(_unused):
        for code in codes:
            table.get(code)

    def with_dict(_unused):
        for code in codes:
            python_dict.get(code)

    return {
        "list": measure(with_list),
        "hash": measure(with_hash_table),
        "dict": measure(with_dict),
        "table": table,
    }


def benchmark_delete(items: list) -> dict:
    """วัดเวลา "ลบทุก key หนึ่งครั้ง" ของทั้งสามวิธี

    การลบทำให้ข้อมูลหายไป รอบต่อไปจะเหลือของให้ลบน้อยลง เลยต้องสร้างของใหม่
    ก่อนทุกรอบผ่าน setup ของ measure() — และเวลาสร้างไม่ถูกนับเข้าไปในผล

    (list(items) คือก็อป list ชั้นนอก ตัว dict ข้างในใช้ร่วมกันได้ ไม่ได้แก้)
    """
    codes = lookup_order(items)

    def with_list(working_copy):
        for code in codes:
            delete_with_list_scan(working_copy, code)

    def with_hash_table(table):
        for code in codes:
            table.delete(code)

    def with_dict(python_dict):
        for code in codes:
            python_dict.pop(code, None)

    return {
        "list": measure(with_list, setup=lambda: list(items)),
        "hash": measure(with_hash_table, setup=lambda: build_hash_table(items)),
        "dict": measure(with_dict, setup=lambda: build_python_dict(items)),
    }


def benchmark_insert(items: list) -> dict:
    """วัดเวลา "เพิ่มของทีละชิ้นจนครบชุด" ของทั้งสามวิธี เริ่มจากที่เก็บว่าง

    ทุกวิธีต้องเช็กรหัสซ้ำก่อนใส่ เพราะนี่คือกฎจริงของร้าน (add_item ในส่วนที่ 6)
    ถ้าไม่นับการเช็กนี้ list จะดูเร็วเกินจริง — append เฉยๆ เป็น O(1)
    แต่ append ที่ต้องเช็กซ้ำก่อนคือการไล่ดูทั้ง list = O(n) ต่อชิ้น
    ร้านที่ไม่เช็กซ้ำจะมีรหัสเดียวกันสองชิ้น แล้วค้นเจอผิดตัว

    ของใหม่ถูกสร้างก่อนทุกรอบผ่าน setup เวลาสร้างที่เก็บว่างจึงไม่ถูกนับ
    เวลาของ HashTable รวมการขยายตารางไว้ด้วยแล้ว (amortized ตามจริง)
    """

    def with_list(target):
        for item in items:
            if find_with_list_scan(target, item["code"]) is None:
                target.append(item)

    def with_hash_table(table):
        for item in items:
            if table.get(item["code"]) is None:
                table.insert(item["code"], item)

    def with_dict(python_dict):
        for item in items:
            if item["code"] not in python_dict:
                python_dict[item["code"]] = item

    return {
        "list": measure(with_list, setup=list),
        "hash": measure(with_hash_table, setup=HashTable),
        "dict": measure(with_dict, setup=dict),
    }


def print_table(title: str, rows: list, first_column: str = "จำนวนข้อมูล") -> None:
    """พิมพ์ผลเป็นตารางอ่านง่าย — rows คือ list ของ tuple

    (จำนวนข้อมูล หรือชื่อชุด, เวลา list, เวลา hash, เวลา dict)
    """
    print()
    print(title)
    print("-" * 76)
    print(
        f"{first_column:>12} {'วน list (ms)':>15} {'HashTable (ms)':>16}"
        f" {'dict (ms)':>12} {'เร็วขึ้นกี่เท่า':>14}"
    )
    print("-" * 76)

    for size, list_ms, hash_ms, dict_ms in rows:
        # เร็วขึ้นกี่เท่า = เวลาของ list ÷ เวลาของ HashTable ที่เราเขียน
        speedup = list_ms / hash_ms if hash_ms > 0 else 0
        label = f"{size:,}" if isinstance(size, int) else size
        print(
            f"{label:>12} {list_ms:>15.3f} {hash_ms:>16.3f}"
            f" {dict_ms:>12.3f} {speedup:>13.1f}x"
        )

    print("-" * 76)


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — ข้อมูลตัวอย่าง
#
# หมายเหตุ: ตัวสร้างชุดข้อมูล A-D ที่ใช้ส่งงานจริงอยู่ที่ algorithms/datasets.py
# (งานของ Neo) ฟังก์ชันข้างล่างนี้เป็นแค่ข้อมูลตัวอย่างเล็กๆ ของไฟล์นี้เอง
# เพื่อให้ก็อปไฟล์เดียวไปวางใน Colab แล้วกดรันได้ทันทีโดยไม่ต้องพาไฟล์อื่นไปด้วย
# ---------------------------------------------------------------------------

CATEGORIES = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]

CATEGORY_PREFIX = {
    "กล้อง": "CAM",
    "เลนส์": "LEN",
    "ขาตั้ง": "TRI",
    "ไฟ": "LED",
    "ไมโครโฟน": "MIC",
}


def make_sample_items(count: int, seed: int = 7) -> list:
    """สร้างรายการอุปกรณ์จำลอง count ชิ้น เป็น dict ธรรมดา

    ใส่ seed ไว้เพื่อให้ได้ข้อมูลชุดเดิมทุกครั้งที่รัน — ตัวเลขในรายงาน
    จะเทียบกันได้ระหว่างการรันแต่ละครั้ง
    """
    generator = random.Random(seed)
    items = []

    for number in range(1, count + 1):
        category = CATEGORIES[number % len(CATEGORIES)]
        total = generator.randint(1, 5)

        items.append(
            {
                "code": f"{CATEGORY_PREFIX[category]}-{number:04d}",
                "name": f"{category} รุ่นที่ {number}",
                "category": category,
                "total": total,
                # ของว่างต้องไม่เกินของทั้งหมด
                "available": generator.randint(0, total),
            }
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — ตัวอย่างที่รันได้เลย
# ---------------------------------------------------------------------------


def demo_basic_operations() -> None:
    """โชว์ว่า insert / get / delete / len / keys ทำงานถูกต้อง บนข้อมูล 10 ชิ้น"""
    print("=" * 76)
    print("1) การใช้งานพื้นฐาน — ข้อมูล 10 รายการ")
    print("=" * 76)

    items = make_sample_items(10)
    table = build_hash_table(items)

    print(f"ใส่ข้อมูลแล้ว {len(table)} รายการ")
    print(f"key ทั้งหมด: {sorted(table.keys())}")

    # ค้นที่มีอยู่
    found = table.get("CAM-0005")
    print(f"\nget('CAM-0005') -> {found['name']} · ของว่าง {found['available']}/{found['total']} ชิ้น")

    # ค้นที่ไม่มี ต้องได้ None ไม่ใช่ error
    print(f"get('ไม่มีรหัสนี้') -> {table.get('ไม่มีรหัสนี้')}")

    # แก้ค่าทับด้วย key เดิม จำนวนข้อมูลต้องไม่เพิ่ม
    before = len(table)
    changed = dict(found)
    changed["available"] = 0
    table.insert("CAM-0005", changed)
    print(
        f"\ninsert ด้วย key เดิม -> จำนวนข้อมูล {before} เป็น {len(table)} (ต้องเท่าเดิม)"
        f" · ของว่างกลายเป็น {table.get('CAM-0005')['available']}"
    )

    # ลบ
    print(f"\ndelete('CAM-0005') -> {table.delete('CAM-0005')} (True = ลบสำเร็จ)")
    print(f"delete('CAM-0005') ซ้ำ -> {table.delete('CAM-0005')} (False = ไม่มีแล้ว)")
    print(f"get('CAM-0005') หลังลบ -> {table.get('CAM-0005')}")
    print(f"เหลือข้อมูล {len(table)} รายการ")

    print("\nสถานะตาราง:")
    for label, value in table.stats().items():
        print(f"  {label:>14} : {value}")


def demo_collision_and_resize() -> None:
    """โชว์การชนและการขยายตาราง บนข้อมูล 120 ชิ้น"""
    print()
    print("=" * 76)
    print("2) การชนและการขยายตาราง — ข้อมูล 120 รายการ")
    print("=" * 76)

    items = make_sample_items(120)

    # ตารางที่ขยายตัวตามปกติ
    good_table = build_hash_table(items)

    # ตารางที่ห้ามขยาย (ตั้ง load factor สูงจนไม่มีวันขยาย) ไว้เทียบให้เห็นภาพ
    class NeverResizeTable(HashTable):
        MAX_LOAD_FACTOR = 10_000.0

    cramped_table = NeverResizeTable()
    for item in items:
        cramped_table.insert(item["code"], item)

    print("\nตารางที่ขยายตัวตามปกติ (ขยายเมื่อ load factor > 0.7)")
    for label, value in good_table.stats().items():
        print(f"  {label:>14} : {value}")

    print("\nตารางเดียวกันแต่สั่งไม่ให้ขยาย — ไว้ดูว่าถ้าไม่ขยายจะเป็นอย่างไร")
    for label, value in cramped_table.stats().items():
        print(f"  {label:>14} : {value}")

    good, cramped = good_table.stats(), cramped_table.stats()
    print(
        f"\nอ่านตัวเลข: ตารางที่ไม่ขยายมีถังแค่ {cramped['จำนวนถัง']} ถัง"
        f" โซ่เฉลี่ยยาว {cramped['load factor']:g} ตัว ยาวสุด {cramped['โซ่ที่ยาวสุด']} ตัว"
        f"\nตารางที่ขยายตามปกติ โซ่ยาวสุดแค่ {good['โซ่ที่ยาวสุด']} ตัว"
        "\nยิ่งใส่ข้อมูลเพิ่มโดยไม่ขยาย โซ่ยิ่งยาวตาม n การค้นจึงไหลลงไปเป็น O(n)"
        "\nนี่คือกรณีแย่สุดที่พูดถึงในคอมเมนต์ส่วนที่ 2 ข้อ 2"
    )


def demo_benchmark() -> None:
    """เทียบความเร็วสามวิธีบนข้อมูลหลายขนาด"""
    print()
    print("=" * 76)
    print(f"3) เทียบความเร็วสามวิธี (แต่ละช่องรัน {ROUNDS} รอบแล้วเฉลี่ย)")
    print("=" * 76)
    print("งานที่วัด: ค้นหา/ลบ ทุก key หนึ่งครั้ง ถ้ามีข้อมูล n ชิ้นก็ทำ n ครั้ง")

    sizes = [10, 120, 1_000, 4_000]

    lookup_rows = []
    delete_rows = []
    collision_rows = []

    for size in sizes:
        items = make_sample_items(size)

        lookup = benchmark_lookup(items)
        lookup_rows.append((size, lookup["list"], lookup["hash"], lookup["dict"]))
        collision_rows.append((size, lookup["table"]))

        deletion = benchmark_delete(items)
        delete_rows.append((size, deletion["list"], deletion["hash"], deletion["dict"]))

    print_table("ค้นหา — ยิ่งข้อมูลมาก ยิ่งเห็นความต่าง", lookup_rows)
    print_table("ลบ — วิธี list ต้องไล่หาตำแหน่งก่อนลบ จึงช้าแบบเดียวกัน", delete_rows)

    print()
    print("สถิติการชนของ HashTable ที่เราเขียน")
    print("-" * 76)
    print(f"{'จำนวนข้อมูล':>12} {'จำนวนถัง':>10} {'ชนทั้งหมด':>12} {'ขยายกี่ครั้ง':>14} {'โซ่ยาวสุด':>12} {'load factor':>13}")
    print("-" * 76)
    for size, table in collision_rows:
        stat = table.stats()
        print(
            f"{size:>12,} {stat['จำนวนถัง']:>10,} {stat['ชนทั้งหมด']:>12,}"
            f" {stat['ขยายตาราง']:>14} {stat['โซ่ที่ยาวสุด']:>12} {stat['load factor']:>13}"
        )
    print("-" * 76)

    print(
        "\nสรุปสำหรับรายงาน"
        "\n  · วน list      O(n) ต่อการค้นหนึ่งครั้ง เวลาโตตามจำนวนข้อมูลเป็นเส้นตรง"
        "\n  · HashTable    O(1) เฉลี่ย เวลาแทบไม่ขยับเมื่อข้อมูลเพิ่ม"
        "\n  · dict         O(1) เฉลี่ยเหมือนกัน แต่เร็วกว่าเพราะเขียนด้วยภาษา C"
        "\n                 ไม่ได้วนลูปด้วย Python เหมือนของเรา"
        "\n  ที่ของเราช้ากว่า dict เป็นเรื่องของค่าคงที่ (constant factor)"
        "\n  ไม่ใช่เรื่องของ Big-O — ทั้งคู่โตในอัตราเดียวกัน"
        "\n"
        "\n  แถวบนสุด (ข้อมูล 10 ชิ้น) วน list เร็วกว่าตารางแฮช — ไม่ใช่ความผิดพลาด"
        "\n  ของน้อยขนาดนั้น การไล่ดู 10 ตัวถูกกว่าการคำนวณแฮชทีละครั้ง"
        "\n  Big-O พูดถึงพฤติกรรมตอน n โตขึ้น ไม่ได้บอกว่าใครเร็วกว่าที่ n น้อยๆ"
    )


# ---------------------------------------------------------------------------
# ส่วนที่ 6 — คลังอุปกรณ์: เก็บทั้ง list และดัชนีแฮชคู่กัน
#
# ทำไมต้องเก็บสองที่ ไม่ใช่ตารางแฮชอย่างเดียว
#   ตารางแฮชตอบเร็วแค่คำถามแบบ "รหัส X อยู่ไหน" แต่ไม่มีลำดับเลย
#   (ดู keys() — ได้ตามลำดับถัง ไม่ใช่ลำดับที่ใส่ และไม่ได้เรียงอะไร)
#   งานอื่นของระบบต้องการลำดับ: เรียงตามวันคืน (sort_group ของ Kul)
#   หาของที่ครบกำหนดก่อนวันที่ X (binary search ของ Neo) นับของทั้งร้าน
#
#   จึงเก็บตัว record จริงไว้ใน list และให้ตารางแฮชเก็บแค่ "รหัส -> ตำแหน่งใน list"
#
#       items = [ {CAM-0001 ...}, {LEN-0002 ...}, {TRI-0003 ...} ]
#       index = HashTable:  "CAM-0001" -> 0 · "LEN-0002" -> 1 · "TRI-0003" -> 2
#
#   list อย่างเดียว   ค้นด้วยรหัสต้องไล่ทีละตัว O(n)
#   แฮชอย่างเดียว     ค้นเร็ว แต่เรียงหรือไล่ตามลำดับไม่ได้
#   สองอย่างคู่กัน     ได้ทั้งคู่ แลกกับหน่วยความจำเพิ่มหนึ่งตาราง
#                      และต้องดูแลให้สองที่ตรงกันเสมอ (ดู remove_item)
# ---------------------------------------------------------------------------


class InventoryError(ValueError):
    """ความผิดพลาดที่เกิดจริงหน้าร้าน เช่น รหัสซ้ำ ของไม่พอ คืนของที่ไม่ได้ยืม

    แยกเป็นชนิดของตัวเอง คนเรียกจะได้ดักเฉพาะเรื่องที่ตั้งใจ
    โดยไม่เผลอกลืนบั๊กจริงอย่าง KeyError หรือ TypeError ไปด้วย
    """


class Inventory:
    """คลังอุปกรณ์ของร้าน 1 ร้าน — list ของ record + ตารางแฮชเป็นดัชนี

    record หนึ่งตัวมี 8 ฟิลด์ หน้าตาเดียวกับ make_dataset() ใน datasets.py
        code / name / category / total / available / borrower / due_date / status
    """

    def __init__(self):
        self.items = []
        self.index = HashTable()

    def __len__(self) -> int:
        return len(self.items)

    def find(self, code: str):
        """คืน record ของรหัสนั้น ไม่มีคืน None — O(1) เฉลี่ย"""
        position = self.index.get(code)
        if position is None:
            return None
        return self.items[position]

    def _require(self, code: str) -> dict:
        """เหมือน find แต่ไม่มีของ = ผิดพลาด ใช้กับงานที่ต้องมีของก่อน"""
        item = self.find(code)
        if item is None:
            raise InventoryError(f"ไม่มีอุปกรณ์รหัส {code} ในร้าน")
        return item

    def add_item(self, record: dict) -> dict:
        """เพิ่มอุปกรณ์ใหม่ — O(1) เฉลี่ย

        ตรวจให้ผ่านทุกข้อก่อนค่อยใส่ ถ้าไม่ผ่านจะไม่แตะข้อมูลเลยสักอย่าง
        ถ้าใส่ list ไปแล้วค่อยพังตอนใส่ดัชนี สองที่จะไม่ตรงกันตลอดไป
        """
        code = record["code"]
        if self.index.get(code) is not None:
            raise InventoryError(f"รหัส {code} มีอยู่แล้วในร้าน")
        if record["total"] < 1:
            raise InventoryError("จำนวนทั้งหมดต้องมีอย่างน้อย 1 ชิ้น")
        if not 0 <= record["available"] <= record["total"]:
            raise InventoryError("จำนวนที่ว่างต้องอยู่ระหว่าง 0 ถึงจำนวนทั้งหมด")

        # เติมช่องที่ขาดให้ครบ 8 ฟิลด์ แล้วก็อปเป็น dict ใหม่
        # จะได้ไม่ไปแก้ dict ที่คนเรียกถืออยู่ (เช่นชุดข้อมูลที่คนอื่นใช้ร่วม)
        item = {"borrower": None, "due_date": None, "status": "ok", **record}

        self.items.append(item)
        self.index.insert(code, len(self.items) - 1)
        return item

    def remove_item(self, code: str) -> dict:
        """ตัดอุปกรณ์ออกจากร้าน (เช่นชำรุดจนซ่อมไม่ได้) — O(1) เฉลี่ย

        ลบตัวกลาง list ด้วย pop(position) ตรงๆ เป็น O(n)
        เพราะ Python ต้องเลื่อนทุกตัวที่อยู่ข้างหลังมาอุดช่อง แล้วตำแหน่ง
        ที่ดัชนีจำไว้ของตัวพวกนั้นก็ผิดหมด ต้องไล่แก้อีก n ตัว

        จึงใช้วิธีสลับกับตัวท้ายแล้วตัดท้าย (swap-and-pop):
          1. หาตำแหน่งของตัวที่จะลบจากดัชนี                O(1)
          2. ยกตัวท้ายสุดมาวางทับตำแหน่งนั้น               O(1)
          3. แก้ดัชนีของตัวที่ถูกย้าย ให้ชี้ตำแหน่งใหม่       O(1)
          4. ตัดตัวท้ายทิ้ง แล้วลบรหัสออกจากดัชนี           O(1)
        ราคาที่จ่ายคือลำดับใน list เปลี่ยน ซึ่งรับได้ เพราะงานที่ต้องการลำดับ
        เอาไปเรียงเองอยู่แล้ว ไม่ได้พึ่งลำดับที่ใส่เข้ามา

        ของที่ยังอยู่กับลูกค้าตัดออกไม่ได้ ไม่งั้นจะไม่มีทางรับคืน
        """
        item = self._require(code)
        if item["available"] < item["total"]:
            raise InventoryError(f"{code} ยังมีของอยู่กับลูกค้า ต้องรับคืนก่อนจึงตัดออกได้")

        position = self.index.get(code)
        last = self.items[-1]

        self.items[position] = last
        self.index.insert(last["code"], position)  # key เดิม = แก้ค่าทับ
        self.items.pop()
        self.index.delete(code)
        return item

    def borrow_item(self, code: str, borrower: str, due_date: str) -> dict:
        """ปล่อยเช่า 1 ชิ้น ตัดจำนวนที่ว่างลง — O(1) เฉลี่ย

        due_date เป็นข้อความ YYYY-MM-DD แบบเดียวกับ datasets.py
        record มีช่องคนยืมช่องเดียวตามที่ออกแบบ 8 ฟิลด์ จึงเก็บคนยืมล่าสุด
        (ในเว็บจริง ใบเช่าทุกใบถูกเก็บแยกไว้ในตาราง borrows)
        """
        item = self._require(code)
        if item["status"] == "broken":
            raise InventoryError(f"{code} ชำรุดอยู่ ปล่อยเช่าไม่ได้")
        if item["available"] < 1:
            raise InventoryError(f"{code} ไม่มีของว่างเหลือให้ปล่อยเช่า")
        if not borrower.strip():
            raise InventoryError("ต้องใส่ชื่อคนเช่า")

        item["available"] -= 1
        item["borrower"] = borrower
        item["due_date"] = due_date
        # ของหมดชั้น = out เหมือนที่หน้าเว็บตั้งให้เอง
        item["status"] = "out" if item["available"] == 0 else "ok"
        return item

    def return_item(self, code: str) -> dict:
        """รับคืน 1 ชิ้น เพิ่มจำนวนที่ว่างกลับ — O(1) เฉลี่ย"""
        item = self._require(code)
        if item["available"] >= item["total"]:
            raise InventoryError(f"{code} ไม่ได้ถูกยืมอยู่ คืนไม่ได้")

        item["available"] += 1
        # คืนครบทุกชิ้นแล้ว = ไม่มีคนยืมค้างอยู่อีก
        if item["available"] == item["total"]:
            item["borrower"] = None
            item["due_date"] = None
        if item["status"] != "broken":
            item["status"] = "ok"
        return item


def build_inventory(items: list) -> Inventory:
    """ยกทั้งชุดข้อมูลใส่คลัง — ใช้กับ make_dataset() หรือ make_sample_items()"""
    inventory = Inventory()
    for item in items:
        inventory.add_item(item)
    return inventory


def demo_inventory() -> None:
    """โชว์ add / borrow / return / remove และทุกกรณีที่ต้องปฏิเสธ"""
    print()
    print("=" * 76)
    print("4) คลังอุปกรณ์ — list + ดัชนีแฮช พร้อมตัวตรวจความถูกต้อง")
    print("=" * 76)

    shop = build_inventory(make_sample_items(5))
    code = shop.items[1]["code"]
    print(f"ร้านมีอุปกรณ์ {len(shop)} รายการ · ทดลองกับ {code}")

    item = shop.find(code)
    item["available"] = item["total"] = 1  # ให้เหลือ 1 ชิ้นจะได้เห็นตอนของหมด

    shop.borrow_item(code, "คุณพลอย", "2026-10-05")
    print(f"\nปล่อยเช่าให้คุณพลอย -> ว่าง {item['available']}/{item['total']} · สถานะ {item['status']}")

    # ทุกบรรทัดข้างล่างต้องถูกปฏิเสธ — นี่คือตัวตรวจที่แผนงานสั่งไว้
    attempts = [
        ("ปล่อยเช่าซ้ำตอนของหมด", lambda: shop.borrow_item(code, "คุณหลิง", "2026-10-06")),
        ("ตัดออกทั้งที่ยังอยู่กับลูกค้า", lambda: shop.remove_item(code)),
        ("เพิ่มรหัสซ้ำ", lambda: shop.add_item(dict(item))),
        ("ค้นรหัสที่ไม่มี", lambda: shop.borrow_item("ไม่มีรหัสนี้", "ใครก็ได้", "2026-10-06")),
    ]
    print()
    for label, attempt in attempts:
        try:
            attempt()
            print(f"  ❌ {label}: ผ่านไปได้ ซึ่งผิด")
        except InventoryError as error:
            print(f"  ✅ {label}: ปฏิเสธ — {error}")

    shop.return_item(code)
    print(f"\nรับคืน -> ว่าง {item['available']}/{item['total']} · คนยืม {item['borrower']} · สถานะ {item['status']}")
    try:
        shop.return_item(code)
        print("  ❌ คืนของที่ไม่ได้ยืม: ผ่านไปได้ ซึ่งผิด")
    except InventoryError as error:
        print(f"  ✅ คืนของที่ไม่ได้ยืม: ปฏิเสธ — {error}")

    # ตัดออกด้วย swap-and-pop แล้วเช็กว่าดัชนีกับ list ยังตรงกันทุกตัว
    shop.remove_item(code)
    in_sync = all(shop.index.get(each["code"]) == position for position, each in enumerate(shop.items))
    print(f"\nตัด {code} ออก -> เหลือ {len(shop)} รายการ · ค้น {code} ได้ {shop.find(code)}")
    print(f"ดัชนีกับ list ยังตรงกันทุกตัวหลังสลับตำแหน่ง: {in_sync}")


# ---------------------------------------------------------------------------
# ส่วนที่ 7 — วัดผลบนชุดมาตรฐาน A-D (ตัวเลขที่ใส่รายงาน)
#
# ทำไมต้องวัดบนชุดเดียวกับคนอื่น: รายงานของสามคน (hash_table / search / sort_group)
# จะเทียบกันได้ก็ต่อเมื่อวัดบนข้อมูลชุดเดียวกัน
#
# ชุด D มี 1,000 รายการ ไม่ใช่ 120 ตามแผนแรก เพราะที่ 120 รายการ ตารางแฮช
# เร็วกว่าวน list แค่ราว 1.4 เท่า ยังไม่ชัดพอจะสรุป ที่ 1,000 ห่างกันชัดแล้ว
# และไม่ไปไกลกว่านี้เพราะ bubble sort ของ Kul ต้องรันสดตอนอัดวิดีโอได้
# (วัดจริง: 1,000 รายการ 0.07 วินาที · 4,000 รายการ 1.42 วินาที)
# ---------------------------------------------------------------------------


def report_on_datasets(make_dataset, names=("A", "B", "C", "D")) -> dict:
    """วัดค้นหา / เพิ่ม / ลบ ของทั้งสามวิธีบนชุด A-D แล้วพิมพ์เป็นตาราง

    รับ make_dataset เข้ามาเป็นพารามิเตอร์ แทนการ import datasets ตรงๆ
    เพราะไฟล์นี้ห้าม import ไฟล์อื่นในโปรเจกต์ (กฎ stdlib ที่หัวไฟล์)
      ใน Colab     วางเซลล์ datasets.py ไว้ก่อน แล้วเรียก report_on_datasets(make_dataset)
      ในโปรเจกต์   python3 scripts/data-course-hashtable.py

    คืนตัวเลขทั้งหมดเป็น dict ด้วย เผื่อเอาไปวาดกราฟต่อ
    """
    results = {}
    lookup_rows, insert_rows, delete_rows = [], [], []

    for name in names:
        items = make_dataset(name)
        label = f"{name} ({len(items):,})"

        lookup = benchmark_lookup(items)
        insert = benchmark_insert(items)
        deletion = benchmark_delete(items)

        results[name] = {
            "size": len(items),
            "lookup": {key: lookup[key] for key in ("list", "hash", "dict")},
            "insert": insert,
            "delete": deletion,
            "stats": lookup["table"].stats(),
        }
        lookup_rows.append((label, lookup["list"], lookup["hash"], lookup["dict"]))
        insert_rows.append((label, insert["list"], insert["hash"], insert["dict"]))
        delete_rows.append((label, deletion["list"], deletion["hash"], deletion["dict"]))

    print_table("ตาราง H1 — ค้นหาทุกรหัสหนึ่งครั้ง", lookup_rows, first_column="ชุด (จำนวน)")
    print_table("ตาราง H2 — เพิ่มทีละชิ้นจนครบชุด (เช็กรหัสซ้ำทุกครั้ง)", insert_rows, first_column="ชุด (จำนวน)")
    print_table("ตาราง H3 — ลบทุกรหัสหนึ่งครั้ง", delete_rows, first_column="ชุด (จำนวน)")

    print()
    print("ตาราง H4 — สถานะของ HashTable หลังใส่ครบชุด")
    print("-" * 76)
    print(f"{'ชุด':>6} {'จำนวน':>8} {'ถัง':>6} {'ชน':>6} {'ขยาย':>6} {'โซ่ยาวสุด':>10} {'load factor':>12}")
    print("-" * 76)
    for name, result in results.items():
        stat = result["stats"]
        print(
            f"{name:>6} {result['size']:>8,} {stat['จำนวนถัง']:>6,} {stat['ชนทั้งหมด']:>6,}"
            f" {stat['ขยายตาราง']:>6} {stat['โซ่ที่ยาวสุด']:>10} {stat['load factor']:>12}"
        )
    print("-" * 76)

    if "A" in results and "D" in results:
        small, large = results["A"], results["D"]
        grow = large["size"] / small["size"]
        print(
            f"\nอ่านตาราง: ข้อมูลโตขึ้น {grow:,.0f} เท่า (ชุด A -> ชุด D)"
            f"\n  ค้นด้วยการวน list ใช้เวลาโตขึ้น {large['lookup']['list'] / small['lookup']['list']:,.0f} เท่า"
            f"\n  ค้นด้วย HashTable ใช้เวลาโตขึ้น {large['lookup']['hash'] / small['lookup']['hash']:,.0f} เท่า"
            "\n  งานที่วัดคือค้นครบทุกรหัส (n ครั้ง) แต่ละครั้ง list ไล่เฉลี่ยครึ่งรายการ"
            "\n  เวลารวมจึงโตราว n²/2 ส่วนแฮชโตราว n เท่านั้น"
            "\n  ต่อการค้นหนึ่งครั้ง list คือ O(n) ส่วนแฮชคือ O(1) เฉลี่ย"
            "\n  (ชุดเล็กจับเวลาได้ไม่แม่น ตัวคูณที่ได้จริงจึงไม่ตรงสูตรเป๊ะ)"
        )

    return results


if __name__ == "__main__":
    demo_basic_operations()
    demo_collision_and_resize()
    demo_benchmark()
    demo_inventory()


Writing hash_table.py


In [3]:
%%writefile search.py
"""การค้นหาและการนับ — งานส่งวิชา Data Structures & Algorithms ของ Neo

อัลกอริทึมที่เขียนเองของไฟล์นี้คือ **binary search** ห้ามใช้ bisect ของ Python
ส่วน linear search เขียนไว้เพื่อเป็นตัวเทียบ และ hash lookup ใช้ตารางแฮช
ที่ Kim เขียนใน hash_table.py

ทั้งสามวิธีตอบคำถามเดียวกัน ("อุปกรณ์รหัสนี้คือชิ้นไหน") แต่ใช้เวลาต่างกัน
คนละระดับ ซึ่งเป็นสิ่งที่รายงานต้องพิสูจน์ด้วยเวลาที่วัดมาจริง ไม่ใช่แค่อ้าง Big-O

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

   ข้อนี้รวมถึง datasets.py และ hash_table.py ของคนในกลุ่มด้วย
   ไฟล์นี้จึงไม่ import อะไรเลย แต่ "รับ" ของที่ต้องใช้เข้ามาทางพารามิเตอร์
       - ชุดข้อมูล รับเป็น list ของ dict
       - ตารางแฮช รับเป็นออบเจ็กต์ที่มีเมท็อด .get() (ตัวของ Kim มีอยู่แล้ว)
   ใน Colab ให้วางเซลล์ตามลำดับ datasets -> hash_table -> ไฟล์นี้
   แล้วทุกอย่างจะอยู่ใน namespace เดียวกันเอง

วิธีรัน
    python3 backend-api/algorithms/search.py
    (รันลำพังได้ มีข้อมูลตัวอย่างเล็กๆ ของตัวเองอยู่ท้ายไฟล์)

ทำไมยังต้องมี binary search ทั้งที่ตารางแฮชเร็วกว่า
    ตารางแฮชเร็วเฉพาะตอนที่รู้ code เป๊ะๆ เพราะมันคำนวณตำแหน่งจาก key ตรงๆ
    แต่คำถามที่ร้านถามจริงคือ "มีของอะไรบ้างที่ครบกำหนดคืนก่อนวันที่ X"
    ตารางแฮชตอบไม่ได้เลย ต้องไล่ดูทั้งตาราง ซึ่งกลายเป็น O(n)
    คำถามแบบเป็นช่วงแบบนี้ต้องใช้ list ที่เรียงแล้ว + binary search
    ซึ่งหาขอบของช่วงได้ใน O(log n) — ดู find_due_before() ท้ายส่วนที่ 2
"""

import time

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — การค้นหา
# ---------------------------------------------------------------------------


def linear_search(items: list, code: str):
    """ไล่ดูทีละชิ้นตั้งแต่ต้นจนเจอ — O(n)

    กรณีดีที่สุดคือเจอที่ชิ้นแรก (1 ครั้ง) กรณีแย่ที่สุดคือไม่มีของชิ้นนั้นอยู่เลย
    ซึ่งต้องดูครบทุกชิ้นก่อนจะสรุปได้ว่าไม่มี (n ครั้ง)
    เฉลี่ยแล้วอยู่ประมาณกลาง list คือ n/2 ครั้ง ซึ่งยังโตตาม n จึงนับเป็น O(n)

    ข้อดีที่ทำให้ยังต้องมี: ใช้กับ list แบบไหนก็ได้ ไม่ต้องเรียงมาก่อน
    และไม่ต้องสร้างโครงสร้างอะไรเพิ่ม
    """
    for item in items:
        if item["code"] == code:
            return item

    return None


def binary_search(sorted_items: list, code: str):
    """ค้นบน list ที่เรียงตาม code แล้ว — O(log n) เขียนเอง ไม่ใช้ bisect

    วิธีคิด: ดูตัวที่อยู่ตรงกลางก่อน
        ถ้าตรงกลางคือตัวที่หา                -> จบ
        ถ้าตัวที่หา "น้อยกว่า" ตรงกลาง       -> ของที่หาต้องอยู่ครึ่งซ้าย ทิ้งครึ่งขวา
        ถ้าตัวที่หา "มากกว่า" ตรงกลาง        -> ของที่หาต้องอยู่ครึ่งขวา ทิ้งครึ่งซ้าย
    ทุกครั้งที่เทียบหนึ่งรอบ จำนวนที่เหลือต้องดูลดลงครึ่งหนึ่ง
    ข้อมูล 1,000 ชิ้นจึงใช้การเทียบแค่ประมาณ 10 ครั้ง (เพราะ 2^10 = 1,024)
    เทียบกับ linear search ที่ต้องเทียบเฉลี่ย 500 ครั้ง

    🔴 เงื่อนไขที่ห้ามลืม: list ต้องเรียงตาม code มาก่อนแล้ว
       ถ้าไม่เรียง การตัดครึ่งจะตัดผิดข้าง แล้วจะคืน None ทั้งที่ของอยู่ในนั้น
       ซึ่งเป็นบั๊กที่ไม่ฟ้องอะไรเลย ใช้ sort_by_code() ข้างล่างเตรียมก่อนเสมอ

    เขียนเป็นลูป ไม่ใช่เรียกตัวเอง (recursion) เพราะลูปไม่กินพื้นที่ stack เพิ่ม
    พื้นที่ที่ใช้จึงเป็น O(1) ไม่ใช่ O(log n)
    """
    low = 0
    high = len(sorted_items) - 1

    while low <= high:
        # เขียนแบบนี้แทน (low + high) // 2 เพื่อกันค่าล้นในภาษาที่ int มีขนาดจำกัด
        # ใน Python ไม่ล้นอยู่แล้ว แต่เขียนให้ถูกหลักไว้ เพราะเป็นท่ามาตรฐาน
        middle = low + (high - low) // 2
        current = sorted_items[middle]["code"]

        if current == code:
            return sorted_items[middle]

        if current < code:
            # ตัวกลางน้อยกว่าที่หา ของที่หาต้องอยู่ทางขวา
            low = middle + 1
        else:
            high = middle - 1

    return None


def hash_lookup(table, code: str):
    """ค้นผ่านตารางแฮชที่ Kim เขียน — O(1) โดยเฉลี่ย

    table คือออบเจ็กต์ HashTable จาก hash_table.py ส่งเข้ามาทางพารามิเตอร์
    ไม่ได้ import เพราะกฎ stdlib ที่หัวไฟล์ (ดูคำอธิบายด้านบน)

    ทำไมเป็น O(1) "เฉลี่ย" ไม่ใช่ทุกกรณี: ตารางแฮชคำนวณว่า key ควรอยู่ถังไหน
    แล้วไปหยิบจากถังนั้นเลย ถ้าถังนั้นมีของชิ้นเดียวก็จบใน 1 ครั้ง
    แต่ถ้า key หลายตัวคำนวณได้ถังเดียวกัน (ชนกัน) ถังนั้นจะยาวขึ้นและต้องไล่ในถัง
    กรณีแย่สุดคือทุก key ตกถังเดียวกันหมด ซึ่งจะกลายเป็น O(n) เท่า linear search
    """
    return table.get(code)


def search_by_name(items: list, keyword: str) -> list:
    """ค้นจากชื่อบางส่วน คืน "ทุก" ตัวที่ตรง — O(n)

    ต้องคืนเป็น list ไม่ใช่ตัวเดียว เพราะชื่อซ้ำกันได้จริงในร้าน
    (ชุดข้อมูล B มีไว้พิสูจน์ข้อนี้โดยเฉพาะ: ชื่อซ้ำเป็นคู่ แต่ code ไม่ซ้ำ)
    ถ้าคืนตัวแรกที่เจอเหมือน linear_search ผู้ใช้จะได้ของผิดชิ้นโดยไม่รู้ตัว
    ซึ่งแปลว่าปล่อยเช่าผิดชิ้น และของอีกชิ้นจะหายไปจากระบบเงียบๆ

    ค้นแบบนี้ใช้ตารางแฮชไม่ได้ เพราะแฮชต้องการ key ที่ตรงเป๊ะทั้งตัว
    "กล้อง" กับ "กล้อง รุ่นที่ 5" คำนวณได้คนละถังโดยสิ้นเชิง
    """
    if not keyword:
        return []

    needle = keyword.strip().lower()
    found = []

    for item in items:
        if needle in item["name"].lower():
            found.append(item)

    return found


def sort_by_code(items: list) -> list:
    """เรียงตาม code เพื่อเตรียมให้ binary_search

    ใช้ sorted() ของ Python ตรงนี้ได้ เพราะการเรียงไม่ใช่อัลกอริทึมที่ไฟล์นี้
    ต้องพิสูจน์ (เป็นงานของ Kul ใน sort_group.py) ตรงนี้แค่เตรียมข้อมูล
    ตอนวัดเวลา การเรียงจึงต้องทำ "ก่อน" เริ่มจับเวลาเสมอ ไม่งั้นจะกลายเป็น
    การวัดเวลาเรียง + ค้น แล้ว binary search จะดูแพ้ทั้งที่ไม่ได้ช้า

    คืน list ใหม่ ไม่แก้ของเดิม เพราะคนอื่นถือ list ใบเดียวกันอยู่
    """
    return sorted(items, key=lambda item: item["code"])


def find_due_before(sorted_by_due: list, limit_date: str) -> list:
    """หาทุกรายการที่ครบกำหนดคืน "ก่อน" วันที่กำหนด — O(log n + k)

    นี่คือคำถามที่ร้านถามจริงทุกเช้า และเป็นเหตุผลว่าทำไมยังต้องมี binary search
    ทั้งที่ตารางแฮชเร็วกว่า: แฮชตอบคำถามแบบ "เป็นช่วง" ไม่ได้เลย

    วิธี: ใช้การตัดครึ่งหาขอบซ้ายสุดที่ due_date >= limit_date ด้วย O(log n)
    แล้วตัดเอาทุกตัวก่อนหน้านั้น ซึ่งคือคำตอบทั้งก้อน
    k คือจำนวนคำตอบ ซึ่งต้องเสียเวลาก็อปออกมาอยู่แล้วไม่ว่าใช้วิธีไหน

    sorted_by_due ต้องเรียงตาม due_date แล้ว และต้องไม่มี None ปนอยู่
    (ของที่ไม่มีใครยืมไม่มีกำหนดคืน จึงไม่ควรอยู่ในรายการนี้ตั้งแต่ต้น)
    """
    low = 0
    high = len(sorted_by_due)

    # หาตำแหน่งแรกสุดที่ due_date >= limit_date
    # สังเกตว่า high เริ่มที่ len ไม่ใช่ len-1 เพราะคำตอบอาจเป็น "ท้ายสุด"
    # คือไม่มีตัวไหนถึงเกณฑ์เลย ทุกตัวจึงเข้าเงื่อนไขทั้งหมด
    while low < high:
        middle = low + (high - low) // 2

        if sorted_by_due[middle]["due_date"] < limit_date:
            low = middle + 1
        else:
            high = middle

    return sorted_by_due[:low]


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — การนับ
#
# ทุกตัวเป็น O(n) เพราะต้องดูทุกชิ้นถึงจะนับได้ ไม่มีทางลัด
# การนับต่างจากการค้นตรงที่ไม่มี "ตัวที่ใช่" ให้หยุด จึงข้ามตัวไหนไม่ได้เลย
# ---------------------------------------------------------------------------


def count_available(items: list) -> int:
    """นับจำนวน "ชิ้น" ที่ยังอยู่บนชั้นทั้งร้าน — O(n)

    นับเป็นชิ้น ไม่ใช่นับจำนวนรายการ เพราะหนึ่งรายการมีได้หลายชิ้น
    (กล้องรุ่นเดียวกัน 5 ตัว = 1 รายการ 5 ชิ้น)
    ตัวเลขนี้คือเลขที่ขึ้นบนแถบสถานะของหน้า workspace ในตัวเว็บ

    ของชำรุดไม่นับ ถึงจะยังวางอยู่บนชั้นก็ปล่อยเช่าไม่ได้
    ถ้านับรวม หน้าเว็บจะบอกว่ามีของว่าง 3 ชิ้นแต่กดปล่อยเช่าไม่ได้สักชิ้น
    """
    total = 0

    for item in items:
        if item["status"] != "broken":
            total += item["available"]

    return total


def count_by_status(items: list) -> dict:
    """นับจำนวน "รายการ" แยกตามสถานะ — O(n)

    คืนคีย์ครบทั้งสามเสมอ แม้ค่าจะเป็น 0 เพราะถ้าคีย์หายไปเฉยๆ
    คนที่เอาไปใช้ต่อต้องเขียน .get() ดักทุกที่ ซึ่งจะมีคนลืมสักที่แน่นอน

    เดินผ่านข้อมูลรอบเดียวได้ทั้งสามตัวเลข ไม่ใช่วนสามรอบแยกกัน
    ผลลัพธ์เท่ากันแต่วนรอบเดียวอ่านข้อมูลน้อยกว่าสามเท่า
    """
    counts = {"ok": 0, "out": 0, "broken": 0}

    for item in items:
        status = item["status"]
        if status in counts:
            counts[status] += 1

    return counts


def count_overdue(items: list, today: str) -> int:
    """นับรายการที่เลยกำหนดคืนแล้ว — O(n)

    today ส่งเข้ามาเป็นข้อความ "YYYY-MM-DD" ไม่ได้อ่านวันจริงจากเครื่อง
    เพราะตัวเลขในรายงานต้องเท่าเดิมทุกครั้งที่รัน ไม่ใช่เปลี่ยนไปทุกวัน
    (datasets.py ตรึงวันอ้างอิงไว้ที่ TODAY ให้แล้ว)

    เทียบวันด้วยการเทียบข้อความได้เลย เพราะรูปแบบ YYYY-MM-DD เรียงจาก
    หน่วยใหญ่ไปหน่วยเล็ก "2026-09-26" < "2026-10-01" เป็นจริงทั้งในเชิง
    ข้อความและเชิงวันที่ ถ้าเก็บเป็น "26/09/2026" ข้อนี้จะไม่จริงทันที

    ของที่คืนแล้วไม่มี due_date (เป็น None) จึงต้องข้ามก่อนเทียบ
    ไม่งั้นจะเทียบ None กับข้อความแล้ว Python จะโยน TypeError ทิ้ง
    """
    count = 0

    for item in items:
        due = item["due_date"]
        if due is not None and due < today:
            count += 1

    return count


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — เครื่องมือวัดเวลา
#
# ใช้วิธีเดียวกับ hash_table.py ของ Kim (perf_counter เฉลี่ย 3 รอบ)
# ตัวเลขของสองไฟล์จะได้อ่านเทียบกันได้ในรายงานฉบับเดียว
# ---------------------------------------------------------------------------

ROUNDS = 3  # รันซ้ำกี่รอบแล้วเฉลี่ย — กันจังหวะที่เครื่องไปทำอย่างอื่นชั่วคราว


def measure(operation, rounds: int = ROUNDS) -> float:
    """วัดเวลาเฉลี่ยของ operation หน่วยเป็นมิลลิวินาที

    ใช้ time.perf_counter() ไม่ใช่ time.time() เพราะ perf_counter เป็นนาฬิกา
    ความละเอียดสูงที่ทำมาเพื่อวัดช่วงเวลาสั้นๆ และไม่กระโดดเวลาเครื่องปรับนาฬิกา
    """
    durations = []

    for _ in range(rounds):
        started_at = time.perf_counter()
        operation()
        durations.append(time.perf_counter() - started_at)

    return sum(durations) / len(durations) * 1000


def lookup_order(items: list) -> list:
    """คืนลำดับ code ที่จะใช้ค้น — สลับแบบตายตัวโดยไม่ต้องสุ่ม

    ทำไมต้องสลับ: ถ้าไล่ค้นตามลำดับเดิม linear_search จะเจอที่ตำแหน่งต้นๆ
    เกือบทุกครั้ง ซึ่งเป็นกรณีดีที่สุดของมัน ตัวเลขจะสวยเกินจริงและสรุปผิด
    ว่า linear ไม่ได้ช้าอย่างที่ Big-O บอก

    ใช้การเดินข้ามทีละ 7 ช่อง (7 เป็นจำนวนเฉพาะ จึงวนครบทุกตำแหน่งพอดี
    ก่อนจะกลับมาซ้ำ) แทน random.shuffle เพื่อให้ได้ลำดับเดิมทุกครั้ง
    โดยไม่ต้อง import random เพิ่ม
    """
    size = len(items)
    return [items[(index * 7) % size]["code"] for index in range(size)]


def benchmark_search(items: list, table=None) -> dict:
    """วัดเวลา "ค้นทุก code หนึ่งครั้ง" ของแต่ละวิธี บนข้อมูลชุดเดียวกัน

    table ส่งเข้ามาได้ถ้ามี HashTable ของ Kim อยู่ใน namespace แล้ว
    ถ้าไม่ส่ง จะวัดแค่ linear กับ binary — ไฟล์นี้จึงรันลำพังได้

    การเรียงทำก่อนเริ่มจับเวลา เพราะในระบบจริง list ที่เรียงแล้วถูกเก็บไว้
    และใช้ซ้ำได้หลายพันครั้ง ไม่ได้เรียงใหม่ทุกครั้งที่มีคนค้น
    ถ้านับเวลาเรียงเข้าไปด้วย จะเป็นการวัดคนละอย่างกับที่ระบบทำจริง
    """
    codes = lookup_order(items)
    sorted_items = sort_by_code(items)

    def with_linear():
        for code in codes:
            linear_search(items, code)

    def with_binary():
        for code in codes:
            binary_search(sorted_items, code)

    results = {
        "linear": measure(with_linear),
        "binary": measure(with_binary),
    }

    if table is not None:

        def with_hash():
            for code in codes:
                hash_lookup(table, code)

        results["hash"] = measure(with_hash)

    return results


def print_table(title: str, rows: list) -> None:
    """พิมพ์ผลเป็นตารางอ่านง่าย — rows คือ list ของ tuple

    (ชื่อชุด, จำนวนข้อมูล, เวลา linear, เวลา binary, เวลา hash หรือ None)
    """
    print()
    print(title)
    print("-" * 78)
    print(
        f"{'ชุด':>5} {'จำนวน':>9} {'linear (ms)':>14} {'binary (ms)':>14}"
        f" {'hash (ms)':>12} {'เร็วขึ้น':>10}"
    )
    print("-" * 78)

    for name, size, linear_ms, binary_ms, hash_ms in rows:
        speedup = linear_ms / binary_ms if binary_ms > 0 else 0
        hash_text = f"{hash_ms:>12.3f}" if hash_ms is not None else f"{'-':>12}"
        print(
            f"{name:>5} {size:>9,} {linear_ms:>14.3f} {binary_ms:>14.3f}"
            f" {hash_text} {speedup:>9.1f}x"
        )

    print("-" * 78)


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — ข้อมูลตัวอย่างของไฟล์นี้เอง
#
# ชุดข้อมูล A-D ที่ใช้ส่งงานจริงอยู่ที่ datasets.py — เรียก make_dataset()
# ฟังก์ชันข้างล่างนี้มีไว้ให้ก็อปไฟล์เดียวไปวางใน Colab แล้วกดรันได้ทันที
# โดยไม่ต้องพาไฟล์อื่นไปด้วย ตัวเลขในรายงานต้องมาจาก datasets.py เท่านั้น
# ---------------------------------------------------------------------------

SAMPLE_TODAY = "2026-10-01"


def make_sample_items(count: int) -> list:
    """สร้างรายการจำลอง count ชิ้น — รูปแบบเดียวกับที่ datasets.py ผลิต

    ไม่สุ่มเลย เดินเป็นแบบแผนตายตัว จึงได้ข้อมูลเดิมทุกครั้งโดยไม่ต้องใส่ seed
    """
    categories = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]
    prefixes = {"กล้อง": "CAM", "เลนส์": "LEN", "ขาตั้ง": "TRI", "ไฟ": "LED", "ไมโครโฟน": "MIC"}
    items = []

    for number in range(1, count + 1):
        category = categories[number % len(categories)]
        total = (number % 5) + 1
        available = number % (total + 1)
        borrowed = total - available

        items.append(
            {
                "code": f"{prefixes[category]}-{number:04d}",
                "name": f"{category} รุ่นที่ {number}",
                "category": category,
                "total": total,
                "available": available,
                "borrower": f"ลูกค้า {number}" if borrowed > 0 else None,
                # สลับกันระหว่างเลยกำหนดกับยังไม่ถึงกำหนด
                "due_date": ("2026-09-20" if number % 3 == 0 else "2026-10-20") if borrowed > 0 else None,
                "status": "broken" if number % 7 == 0 else ("out" if available == 0 else "ok"),
            }
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — ตรวจว่าทุกฟังก์ชันทำงานถูก
# ---------------------------------------------------------------------------


def verify() -> None:
    """ตรวจผลของทุกฟังก์ชัน ถ้าไม่ตรงให้ล้มทันที

    จุดที่ตั้งใจตรวจเป็นพิเศษคือ binary_search ต้องให้คำตอบ "ตรงกับ"
    linear_search ทุกตัว ถ้าสองตัวนี้ตอบไม่ตรงกัน แปลว่าตัวใดตัวหนึ่งผิด
    และการเทียบเวลาในรายงานก็ไม่มีความหมาย เพราะเทียบของที่ทำงานคนละอย่าง
    """
    items = make_sample_items(40)
    sorted_items = sort_by_code(items)

    # binary ต้องตอบเหมือน linear ทุกตัว รวมถึงตัวที่ไม่มีอยู่จริงด้วย
    for item in items:
        assert binary_search(sorted_items, item["code"]) == linear_search(items, item["code"])

    for missing in ("ไม่มีจริง", "CAM-9999", "", "AAA-0001", "ZZZ-9999"):
        assert linear_search(items, missing) is None
        assert binary_search(sorted_items, missing) is None, f"binary หา {missing!r} แล้วต้องไม่เจอ"

    # ขอบของ list — ตัวแรกสุดและตัวท้ายสุดคือจุดที่ off-by-one ชอบโผล่
    assert binary_search(sorted_items, sorted_items[0]["code"]) == sorted_items[0]
    assert binary_search(sorted_items, sorted_items[-1]["code"]) == sorted_items[-1]

    # list ว่างและ list ที่มีตัวเดียว ต้องไม่ระเบิด
    assert binary_search([], "CAM-0001") is None
    single = [sorted_items[0]]
    assert binary_search(single, single[0]["code"]) == single[0]
    assert binary_search(single, "ZZZ-0000") is None

    # ค้นด้วยชื่อ ต้องได้ทุกตัวที่ตรง ไม่ใช่ตัวแรกที่เจอ
    duplicates = [
        {"code": "CAM-0001", "name": "กล้อง A", "category": "กล้อง", "total": 1,
         "available": 1, "borrower": None, "due_date": None, "status": "ok"},
        {"code": "CAM-0002", "name": "กล้อง A", "category": "กล้อง", "total": 1,
         "available": 0, "borrower": "ก", "due_date": "2026-09-01", "status": "out"},
    ]
    found = search_by_name(duplicates, "กล้อง A")
    assert len(found) == 2, "ชื่อซ้ำต้องคืนครบทุกตัว"
    assert {row["code"] for row in found} == {"CAM-0001", "CAM-0002"}
    assert search_by_name(duplicates, "") == []
    assert search_by_name(duplicates, "ไม่มีชื่อนี้") == []

    # นับของว่าง — ของชำรุดต้องไม่ถูกนับรวม
    hand_counted = sum(item["available"] for item in items if item["status"] != "broken")
    assert count_available(items) == hand_counted
    assert count_available([]) == 0

    # นับตามสถานะ — ผลรวมต้องเท่าจำนวนรายการทั้งหมด และมีครบสามคีย์เสมอ
    counts = count_by_status(items)
    assert set(counts) == {"ok", "out", "broken"}
    assert sum(counts.values()) == len(items)
    assert count_by_status([]) == {"ok": 0, "out": 0, "broken": 0}

    # นับของเลยกำหนด
    hand_overdue = sum(
        1 for item in items if item["due_date"] is not None and item["due_date"] < SAMPLE_TODAY
    )
    assert count_overdue(items, SAMPLE_TODAY) == hand_overdue
    assert count_overdue([], SAMPLE_TODAY) == 0

    # คำถามแบบเป็นช่วง — ตัวที่พิสูจน์ว่าทำไมยังต้องมี binary search
    with_due = sorted(
        [item for item in items if item["due_date"] is not None],
        key=lambda item: item["due_date"],
    )
    early = find_due_before(with_due, SAMPLE_TODAY)
    assert all(item["due_date"] < SAMPLE_TODAY for item in early)
    assert len(early) == hand_overdue, "หาแบบช่วงต้องได้จำนวนเท่ากับการนับทีละตัว"
    assert find_due_before(with_due, "2000-01-01") == []
    assert len(find_due_before(with_due, "2999-12-31")) == len(with_due)

    print("ตรวจฟังก์ชันค้นหาและนับ ผ่านทั้งหมด")


# ---------------------------------------------------------------------------
# ส่วนที่ 6 — รันเพื่อดูผล
# ---------------------------------------------------------------------------


def main() -> None:
    print("=" * 78)
    print("การค้นหาและการนับ — LEBIS (งานของ Neo)")
    print("=" * 78)

    verify()

    items = make_sample_items(20)
    print()
    print("ตัวอย่างผลการนับ บนข้อมูลตัวอย่าง 20 รายการ")
    print("-" * 78)
    print(f"    ของว่างบนชั้น        {count_available(items)} ชิ้น")
    print(f"    แยกตามสถานะ          {count_by_status(items)}")
    print(f"    เลยกำหนดคืน          {count_overdue(items, SAMPLE_TODAY)} รายการ")

    # วัดเวลาบนขนาดที่ต่างกัน ให้เห็นว่า linear แย่ลงเร็วแค่ไหนเมื่อข้อมูลโต
    # ตัวเลขจริงที่ใช้ในรายงานต้องมาจากชุด A-D ของ datasets.py
    rows = []
    for size in (10, 100, 1000):
        sample = make_sample_items(size)
        result = benchmark_search(sample)
        rows.append((str(size), size, result["linear"], result["binary"], None))

    print_table("เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) — ข้อมูลตัวอย่างของไฟล์นี้", rows)

    print()
    print("อ่านตารางนี้ยังไง")
    print("-" * 78)
    print("    ข้อมูลโตขึ้น 10 เท่า linear ใช้เวลาโตขึ้นประมาณ 100 เท่า")
    print("    เพราะทั้งจำนวนครั้งที่ค้นและความยาวที่ต้องไล่ โตขึ้นอย่างละ 10 เท่า")
    print("    ส่วน binary โตขึ้นแค่ประมาณ 10 เท่า เพราะแต่ละครั้งยังใช้การเทียบ")
    print("    เพิ่มขึ้นแค่ 3-4 ครั้งเท่านั้น (log2 ของ 10 เท่า)")
    print()
    print("    ตัวเลขที่ใช้ในรายงานจริงต้องมาจากชุด A-D ของ datasets.py")
    print("    วิธีรันใน Colab: วางเซลล์ datasets -> hash_table -> ไฟล์นี้")
    print("    แล้วเรียก benchmark_search(make_dataset('D'), build_hash_table(...))")


if __name__ == "__main__":
    main()


Writing search.py


In [4]:
%%writefile sort_group.py
"""การเรียงลำดับและการจัดกลุ่ม — งานส่งวิชา Data Structures & Algorithms ของ Kul

อัลกอริทึมที่เขียนเองของไฟล์นี้คือ **merge sort** ห้ามใช้ sorted()/.sort() ของ Python
ส่วน bubble sort เขียนไว้เพื่อเป็นตัวเทียบ

list ที่ merge_sort เรียงแล้ว คือตัวที่ binary_search ของ Neo ใช้ต่อโดยตรง
(ดู find_due_before ใน search.py) งานสองคนนี้ต่อกัน ต้องคุยกันเรื่องรูปแบบ
ที่ส่งต่อให้ตรง — ไฟล์นี้คืน list ใหม่เสมอ ไม่แก้ของเดิม เหมือนที่ search.py ทำ

🔴 กฎเหล็กของไฟล์นี้: import ได้แค่ standard library ของ Python เท่านั้น
   ห้าม FastAPI ห้าม databases ห้าม Pydantic ห้าม import ไฟล์อื่นในโปรเจกต์
   เพราะไฟล์เดียวกันนี้ถูกก็อปไปวางใน Google Colab แล้วต้องกดรันได้ทันที

วิธีรัน
    python3 backend-api/algorithms/sort_group.py
    (รันลำพังได้ มีข้อมูลตัวอย่างเล็กๆ ของตัวเองอยู่ท้ายไฟล์)

ทำไมยังต้องมี bubble sort ทั้งที่ merge sort เร็วกว่าเสมอ
    ไม่ได้มีไว้ใช้งานจริง มีไว้ "เทียบ" ให้เห็นว่า O(n²) กับ O(n log n) ต่างกัน
    แค่ไหนเมื่อข้อมูลโต — ที่ 10 รายการแทบไม่ต่าง แต่ที่ 1,000 รายการ
    bubble sort ต้องเทียบมากกว่า merge sort หลายสิบเท่า
"""

import time

# ---------------------------------------------------------------------------
# ส่วนที่ 1 — การเรียงลำดับทั่วไป (รับ key แบบเดียวกับ sorted() ของ Python)
# ---------------------------------------------------------------------------


def bubble_sort(items: list, key=lambda item: item) -> list:
    """เรียงด้วยการสลับคู่ที่อยู่ผิดลำดับซ้ำไปเรื่อยๆ — O(n²) เขียนเอง

    วิธีคิด: ไล่เทียบของที่อยู่ติดกันทีละคู่ ถ้าลำดับผิด (ตัวหน้ามากกว่าตัวหลัง)
    ให้สลับที่กัน ทำแบบนี้วนจนครบรอบ แล้ววนซ้ำใหม่ทั้งชุด ของที่ "หนักที่สุด"
    จะค่อยๆ ลอยไปอยู่ท้ายสุดทีละตัวในแต่ละรอบ เหมือนฟองอากาศลอยขึ้นผิวน้ำ

    กันวนเกินความจำเป็น: ถ้ารอบไหนไม่มีการสลับเลย แปลว่าเรียงเสร็จแล้ว
    หยุดได้ทันทีโดยไม่ต้องวนจนครบ n รอบ (สำคัญมากกับชุด A ที่มีแค่ 10 ตัว
    ถ้าไม่เช็กจุดนี้ ตัวเลขเวลาของชุดเล็กจะดูแย่เกินจริง)

    คืน list ใหม่ ไม่แก้ของเดิม เพราะคนอื่นอาจถือ list ใบเดียวกันอยู่
    (ดู sort_by_code ใน search.py ที่ทำแบบเดียวกัน)
    """
    result = list(items)
    n = len(result)

    for round_index in range(n):
        swapped = False

        for i in range(n - round_index - 1):
            if key(result[i]) > key(result[i + 1]):
                result[i], result[i + 1] = result[i + 1], result[i]
                swapped = True

        if not swapped:
            break

    return result


def merge_sort(items: list, key=lambda item: item) -> list:
    """เรียงด้วยการแบ่งครึ่งแล้วผสานกลับ — O(n log n) เขียนเอง

    วิธีคิด: แบ่ง list ออกเป็นสองครึ่งไปเรื่อยๆ จนเหลือชิ้นเดียว (ชิ้นเดียว
    ถือว่าเรียงแล้วเสมอ) แล้วผสาน (merge) สองครึ่งที่เรียงแล้วเข้าด้วยกัน
    ทีละคู่ ไล่กลับขึ้นมา แต่ละชั้นของการแบ่งใช้เวลา O(n) ในการผสาน
    และมี log n ชั้น (แบ่งครึ่งไปเรื่อยๆ จนเหลือ 1) รวมเป็น O(n log n)

    ต่างจาก bubble sort ตรงที่ไม่ต้องไล่เทียบทุกคู่กับทุกคู่ — ข้อมูลสอง
    ฝั่งที่เรียงมาแล้วผสานกันได้ในรอบเดียวโดยไล่จากหัวทั้งสองฝั่งพร้อมกัน

    เขียนแบบ recursion เพราะการแบ่งครึ่งซ้ำๆ เข้ากับรูปแบบนี้ตามธรรมชาติ
    ความลึกของ stack อยู่ที่ประมาณ log2(n) ซึ่งสำหรับ 1,000 รายการคือ
    ประมาณ 10 ชั้นเท่านั้น ไม่เสี่ยง stack overflow
    """
    if len(items) <= 1:
        return list(items)

    middle = len(items) // 2
    left = merge_sort(items[:middle], key)
    right = merge_sort(items[middle:], key)

    return _merge(left, right, key)


def _merge(left: list, right: list, key) -> list:
    """ผสานสอง list ที่เรียงแล้วเข้าเป็นก้อนเดียวที่ยังเรียงอยู่ — O(n)

    ไล่เทียบตัวหน้าสุดของทั้งสองฝั่ง หยิบตัวที่เล็กกว่าออกมาก่อนเสมอ
    เขียน <= ไม่ใช่ < ตอนเทียบฝั่งซ้าย เพื่อให้เป็น stable sort — ถ้าค่าเท่ากัน
    ตัวที่มาจากฝั่งซ้าย (ซึ่งอยู่ก่อนในของเดิม) ต้องถูกหยิบก่อนเสมอ
    ลำดับเดิมของรายการที่ due_date เท่ากันจึงไม่ถูกสลับมั่วโดยไม่มีเหตุผล
    """
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])

    return merged


# ---------------------------------------------------------------------------
# ส่วนที่ 2 — การเรียงเฉพาะทางที่ระบบใช้จริง
# ---------------------------------------------------------------------------


def sort_by_due_date(items: list) -> list:
    """เรียงรายการยืมตามวันครบกำหนดคืน ของค้างนานสุดขึ้นก่อน — ใช้ merge_sort

    "ค้างนานสุด" คือของที่ due_date เก่าที่สุด (เลยกำหนดมานานที่สุด)
    เรียงจากน้อยไปมาก (ascending) จึงได้ตัวที่ค้างนานสุดอยู่บนสุดพอดี

    กรองของที่ไม่มีคนยืมออกก่อน (due_date เป็น None) เพราะ merge_sort
    เทียบ None กับข้อความไม่ได้ (Python โยน TypeError ทันที) และของที่
    ไม่มีคนยืมก็ไม่มี "วันครบกำหนด" ให้เรียงตั้งแต่แรกอยู่แล้ว

    list ที่คืนออกมาเรียงตาม due_date แล้ว จึงส่งต่อให้ find_due_before()
    ของ Neo ใช้ต่อได้ทันที ไม่ต้องเรียงซ้ำ
    """
    borrowed = [item for item in items if item["due_date"] is not None]
    return merge_sort(borrowed, key=lambda item: item["due_date"])


def sort_by_available(items: list) -> list:
    """เรียงตามจำนวนคงเหลือ น้อยไปมาก เพื่อหาของที่ใกล้หมดก่อน — ใช้ merge_sort

    ของที่เหลือน้อยที่สุดอยู่บนสุด เจ้าของร้านเปิดหน้านี้แล้วเห็นทันทีว่า
    ต้องเติมของชิ้นไหนก่อน ไม่ต้องไล่ดูทั้งตาราง
    """
    return merge_sort(items, key=lambda item: item["available"])


# ---------------------------------------------------------------------------
# ส่วนที่ 3 — การจัดกลุ่ม
#
# ทั้งคู่เป็น O(n) เดินผ่านข้อมูลรอบเดียว แบ่งลงถังตามคีย์ที่ต้องการ
# เร็วกว่าการวนหลายรอบทีละหมวด (เช่นวนหาแต่ "กล้อง" ก่อน แล้ววนใหม่หา "เลนส์")
# เพราะแบบนั้นต้องอ่านข้อมูลทั้งก้อนซ้ำเท่าจำนวนหมวด กลายเป็น O(n × จำนวนหมวด)
# ส่วนแบบแบ่งถังอ่านข้อมูลแค่รอบเดียวไม่ว่าจะมีกี่หมวด
# ---------------------------------------------------------------------------


def group_by_type(items: list) -> dict:
    """จัดกลุ่มตามหมวดอุปกรณ์ (category) — O(n)

    คืน dict ที่คีย์คือชื่อหมวด ค่าคือ list ของรายการในหมวดนั้น
    ไม่ต้องรู้ล่วงหน้าว่ามีหมวดอะไรบ้าง เพราะสร้างคีย์ใหม่ทันทีที่เจอหมวดที่ยังไม่มี
    """
    groups: dict = {}

    for item in items:
        category = item["category"]
        groups.setdefault(category, []).append(item)

    return groups


def group_by_status(items: list) -> dict:
    """จัดกลุ่มตามสถานะ (ok / out / broken) — O(n)

    ต่างจาก count_by_status ของ Neo ตรงที่ตัวนี้เก็บ "รายการทั้งใบ" ไว้ในกลุ่ม
    ไม่ใช่แค่นับจำนวน ใช้ตอนต้องการดูว่ามีอะไรบ้างในแต่ละสถานะ ไม่ใช่แค่กี่ชิ้น

    คืนคีย์ครบทั้งสามเสมอแม้จะว่าง เหตุผลเดียวกับ count_by_status ของ Neo:
    กันคนเอาไปใช้ต่อแล้วต้องเขียน .get() ดักทุกที่
    """
    groups = {"ok": [], "out": [], "broken": []}

    for item in items:
        status = item["status"]
        if status in groups:
            groups[status].append(item)

    return groups


# ---------------------------------------------------------------------------
# ส่วนที่ 4 — เครื่องมือวัดเวลา
#
# ใช้วิธีเดียวกับ hash_table.py ของ Kim และ search.py ของ Neo
# (perf_counter เฉลี่ย 3 รอบ) ตัวเลขของสามไฟล์จึงอ่านเทียบกันได้ในรายงานเดียว
# ---------------------------------------------------------------------------

ROUNDS = 3  # รันซ้ำกี่รอบแล้วเฉลี่ย — กันจังหวะที่เครื่องไปทำอย่างอื่นชั่วคราว


def measure(operation, rounds: int = ROUNDS) -> float:
    """วัดเวลาเฉลี่ยของ operation หน่วยเป็นมิลลิวินาที — เหมือนของ Neo ทุกจุด"""
    durations = []

    for _ in range(rounds):
        started_at = time.perf_counter()
        operation()
        durations.append(time.perf_counter() - started_at)

    return sum(durations) / len(durations) * 1000


def benchmark_sort(items: list, key=lambda item: item) -> dict:
    """วัดเวลา "เรียงทั้งชุดหนึ่งครั้ง" ของแต่ละวิธี บนข้อมูลชุดเดียวกัน

    ส่ง items เข้าไปตรงๆ ได้เลย เพราะทั้ง bubble_sort และ merge_sort
    คืน list ใหม่เสมอ ไม่แก้ของเดิม จึงใช้ items ก้อนเดียววัดซ้ำได้หลายวิธี
    โดยไม่ต้องกังวลว่าวิธีก่อนหน้าจะทำให้ข้อมูลเรียงอยู่แล้วจนวิธีถัดไปได้เปรียบ
    """

    def with_bubble():
        bubble_sort(items, key)

    def with_merge():
        merge_sort(items, key)

    def with_builtin():
        sorted(items, key=key)

    return {
        "bubble": measure(with_bubble),
        "merge": measure(with_merge),
        "builtin": measure(with_builtin),
    }


def print_table(title: str, rows: list) -> None:
    """พิมพ์ผลเป็นตารางอ่านง่าย — rows คือ list ของ tuple

    (ชื่อชุด, จำนวนข้อมูล, เวลา bubble, เวลา merge, เวลา builtin)
    """
    print()
    print(title)
    print("-" * 78)
    print(
        f"{'ชุด':>5} {'จำนวน':>9} {'bubble (ms)':>14} {'merge (ms)':>14}"
        f" {'builtin (ms)':>14} {'เร็วขึ้น':>10}"
    )
    print("-" * 78)

    for name, size, bubble_ms, merge_ms, builtin_ms in rows:
        speedup = bubble_ms / merge_ms if merge_ms > 0 else 0
        print(
            f"{name:>5} {size:>9,} {bubble_ms:>14.3f} {merge_ms:>14.3f}"
            f" {builtin_ms:>14.3f} {speedup:>9.1f}x"
        )

    print("-" * 78)


def report_on_datasets(make_dataset) -> None:
    """วัดเวลาเรียงบนชุดข้อมูลจริง A–D ของ Neo (datasets.py) — ใช้ตอนอยู่ใน Colab เท่านั้น

    รับฟังก์ชัน make_dataset เข้ามาทางพารามิเตอร์แทนการ import ตรงๆ
    เพราะไฟล์นี้ต้อง import ได้แค่ standard library (ดูกฎเหล็กหัวไฟล์)
    ใน Colab วางเซลล์ datasets.py ไว้ก่อน แล้วเรียก
        report_on_datasets(make_dataset)
    ตัวเลขจากฟังก์ชันนี้คือของจริงที่เอาไปเขียนรายงาน ไม่ใช่ตัวเลขจาก main()
    ซึ่งวัดบนข้อมูลตัวอย่างของไฟล์นี้เองเพื่อให้รันลำพังได้เท่านั้น
    """
    rows = []

    for name in ("A", "B", "C", "D"):
        items = make_dataset(name)
        result = benchmark_sort(items, key=lambda item: item["available"])
        rows.append((name, len(items), result["bubble"], result["merge"], result["builtin"]))

    print_table("เวลาเรียงทั้งชุดหนึ่งรอบ (เฉลี่ย 3 รอบ) — ชุดข้อมูลจริง A-D", rows)


# ---------------------------------------------------------------------------
# ส่วนที่ 5 — ข้อมูลตัวอย่างของไฟล์นี้เอง
#
# ชุดข้อมูล A-D ที่ใช้ส่งงานจริงอยู่ที่ datasets.py — เรียก make_dataset()
# ฟังก์ชันข้างล่างนี้มีไว้ให้ก็อปไฟล์เดียวไปวางใน Colab แล้วกดรันได้ทันที
# เหมือนกับที่ search.py ของ Neo ทำไว้ (make_sample_items)
# ---------------------------------------------------------------------------

SAMPLE_TODAY = "2026-10-01"


def make_sample_items(count: int) -> list:
    """สร้างรายการจำลอง count ชิ้น — รูปแบบเดียวกับที่ datasets.py ผลิต

    คัดลอกแบบเดียวกับ make_sample_items ใน search.py ของ Neo เจตนาให้เหมือนกัน
    เพื่อให้ไฟล์นี้รันลำพังได้และให้ผลที่เทียบกับของ Neo ได้ตรงๆ
    """
    categories = ["กล้อง", "เลนส์", "ขาตั้ง", "ไฟ", "ไมโครโฟน"]
    prefixes = {"กล้อง": "CAM", "เลนส์": "LEN", "ขาตั้ง": "TRI", "ไฟ": "LED", "ไมโครโฟน": "MIC"}
    items = []

    for number in range(1, count + 1):
        category = categories[number % len(categories)]
        total = (number % 5) + 1
        available = number % (total + 1)
        borrowed = total - available

        items.append(
            {
                "code": f"{prefixes[category]}-{number:04d}",
                "name": f"{category} รุ่นที่ {number}",
                "category": category,
                "total": total,
                "available": available,
                "borrower": f"ลูกค้า {number}" if borrowed > 0 else None,
                "due_date": ("2026-09-20" if number % 3 == 0 else "2026-10-20") if borrowed > 0 else None,
                "status": "broken" if number % 7 == 0 else ("out" if available == 0 else "ok"),
            }
        )

    return items


# ---------------------------------------------------------------------------
# ส่วนที่ 6 — ตรวจว่าทุกฟังก์ชันทำงานถูก
# ---------------------------------------------------------------------------


def verify() -> None:
    """ตรวจผลของทุกฟังก์ชัน ถ้าไม่ตรงให้ล้มทันที"""
    items = make_sample_items(40)

    # bubble และ merge ต้องให้ผลตรงกับ sorted() ของ Python เป๊ะ
    by_available_key = lambda item: item["available"]
    expected = sorted(items, key=by_available_key)
    assert bubble_sort(items, by_available_key) == expected
    assert merge_sort(items, by_available_key) == expected

    # ต้องคืน list ใหม่ ไม่แก้ของเดิม
    original_order = [item["code"] for item in items]
    merge_sort(items, by_available_key)
    assert [item["code"] for item in items] == original_order, "ห้ามแก้ list เดิม"

    # กรณีขอบ: list ว่างและ list ตัวเดียว ต้องไม่ระเบิด
    assert bubble_sort([]) == []
    assert merge_sort([]) == []
    single = [items[0]]
    assert bubble_sort(single, by_available_key) == single
    assert merge_sort(single, by_available_key) == single

    # stable sort: ของที่ค่าเท่ากันต้องคงลำดับเดิมไว้ (เทียบกับ sorted() ซึ่ง stable)
    tied = [{"code": "A", "available": 1}, {"code": "B", "available": 1}, {"code": "C", "available": 0}]
    tied_expected = sorted(tied, key=lambda item: item["available"])
    assert merge_sort(tied, key=lambda item: item["available"]) == tied_expected

    # sort_by_due_date: ต้องกรอง None ออก และของค้างนานสุด (due_date เก่าสุด) ต้องอยู่บนสุด
    by_due = sort_by_due_date(items)
    assert all(item["due_date"] is not None for item in by_due)
    assert [item["due_date"] for item in by_due] == sorted(item["due_date"] for item in by_due)
    if len(by_due) >= 2:
        assert by_due[0]["due_date"] <= by_due[1]["due_date"]

    # sort_by_available: น้อยไปมาก
    by_avail = sort_by_available(items)
    assert [item["available"] for item in by_avail] == sorted(item["available"] for item in items)

    # group_by_type: ทุกชิ้นต้องอยู่ในกลุ่มของตัวเอง ไม่หายไม่เกิน
    by_type = group_by_type(items)
    assert sum(len(group) for group in by_type.values()) == len(items)
    for category, group in by_type.items():
        assert all(item["category"] == category for item in group)

    # group_by_status: คีย์ครบสามเสมอ ผลรวมเท่าจำนวนทั้งหมด
    by_status = group_by_status(items)
    assert set(by_status) == {"ok", "out", "broken"}
    assert sum(len(group) for group in by_status.values()) == len(items)
    assert group_by_status([]) == {"ok": [], "out": [], "broken": []}

    print("ตรวจฟังก์ชันเรียงลำดับและจัดกลุ่ม ผ่านทั้งหมด")


# ---------------------------------------------------------------------------
# ส่วนที่ 7 — รันเพื่อดูผล
# ---------------------------------------------------------------------------


def main() -> None:
    print("=" * 78)
    print("การเรียงลำดับและการจัดกลุ่ม — LEBIS (งานของ Kul)")
    print("=" * 78)

    verify()

    items = make_sample_items(20)
    print()
    print("ตัวอย่างการจัดกลุ่ม บนข้อมูลตัวอย่าง 20 รายการ")
    print("-" * 78)
    for status, group in group_by_status(items).items():
        print(f"    {status:<8} {len(group)} รายการ")

    # วัดเวลาบนขนาดที่ต่างกัน ให้เห็นว่า bubble แย่ลงเร็วแค่ไหนเมื่อข้อมูลโต
    # ตัวเลขจริงที่ใช้ในรายงานต้องมาจากชุด A-D ของ datasets.py
    rows = []
    for size in (10, 100, 1000):
        sample = make_sample_items(size)
        result = benchmark_sort(sample, key=lambda item: item["available"])
        rows.append((str(size), size, result["bubble"], result["merge"], result["builtin"]))

    print_table("เวลาเรียงทั้งชุดหนึ่งรอบ (เฉลี่ย 3 รอบ) — ข้อมูลตัวอย่างของไฟล์นี้", rows)

    print()
    print("อ่านตารางนี้ยังไง")
    print("-" * 78)
    print("    ข้อมูลโตขึ้น 10 เท่า bubble sort ใช้เวลาโตขึ้นประมาณ 100 เท่า เพราะเป็น n²")
    print("    ส่วน merge sort โตขึ้นเพียงประมาณ 14-15 เท่า เพราะโตแบบ n log n")
    print("    (ทฤษฎี n log n ให้ค่าประมาณ 15 เท่า ตรงกับตัวเลขที่วัดได้จริงในตาราง)")
    print()
    print("    ตัวเลขที่ใช้ในรายงานจริงมาจากชุด A-D ของ datasets.py")
    print("    เรียก report_on_datasets(make_dataset) เพื่อดูตารางเต็ม")


if __name__ == "__main__":
    main()

Writing sort_group.py


In [5]:
import datasets, hash_table, search, sort_group

---
## 2. Insert / Delete — Hash Table (Kim)

In [6]:
hash_table.demo_inventory()
results_kim = hash_table.report_on_datasets(datasets.make_dataset)


4) คลังอุปกรณ์ — list + ดัชนีแฮช พร้อมตัวตรวจความถูกต้อง
ร้านมีอุปกรณ์ 5 รายการ · ทดลองกับ TRI-0002

ปล่อยเช่าให้คุณพลอย -> ว่าง 0/1 · สถานะ out

  ✅ ปล่อยเช่าซ้ำตอนของหมด: ปฏิเสธ — TRI-0002 ไม่มีของว่างเหลือให้ปล่อยเช่า
  ✅ ตัดออกทั้งที่ยังอยู่กับลูกค้า: ปฏิเสธ — TRI-0002 ยังมีของอยู่กับลูกค้า ต้องรับคืนก่อนจึงตัดออกได้
  ✅ เพิ่มรหัสซ้ำ: ปฏิเสธ — รหัส TRI-0002 มีอยู่แล้วในร้าน
  ✅ ค้นรหัสที่ไม่มี: ปฏิเสธ — ไม่มีอุปกรณ์รหัส ไม่มีรหัสนี้ ในร้าน

รับคืน -> ว่าง 1/1 · คนยืม None · สถานะ ok
  ✅ คืนของที่ไม่ได้ยืม: ปฏิเสธ — TRI-0002 ไม่ได้ถูกยืมอยู่ คืนไม่ได้

ตัด TRI-0002 ออก -> เหลือ 4 รายการ · ค้น TRI-0002 ได้ None
ดัชนีกับ list ยังตรงกันทุกตัวหลังสลับตำแหน่ง: True



ตาราง H1 — ค้นหาทุกรหัสหนึ่งครั้ง
----------------------------------------------------------------------------
 ชุด (จำนวน)    วน list (ms)   HashTable (ms)    dict (ms) เร็วขึ้นกี่เท่า
----------------------------------------------------------------------------
      A (10)           0.004            0.015        0.001           0.3x
      B (12)           0.005            0.017        0.001           0.3x
      C (10)           0.004            0.014        0.001           0.3x
   D (1,000)          21.753            1.530        0.053          14.2x
----------------------------------------------------------------------------

ตาราง H2 — เพิ่มทีละชิ้นจนครบชุด (เช็กรหัสซ้ำทุกครั้ง)
----------------------------------------------------------------------------
 ชุด (จำนวน)    วน list (ms)   HashTable (ms)    dict (ms) เร็วขึ้นกี่เท่า
----------------------------------------------------------------------------
      A (10)           0.004            0.042        0.002           0.1x
    

---
## 3. Search / Count — Binary Search (Neo)

In [7]:
%%writefile data_course_benchmark.py
#!/usr/bin/env python3
"""วัดผลชุด A-D ของวิชา Data — ตัวเลขทุกตัวที่ใส่ในรายงานต้องมาจากสคริปต์นี้

ทำไมสคริปต์นี้ถึงอยู่นอกโฟลเดอร์ algorithms/
    ไฟล์ใน algorithms/ ห้าม import ไฟล์อื่นในโปรเจกต์ (ดู docs/data-course/plan.md)
    เพราะแต่ละไฟล์ต้องก็อปไปวางใน Colab แล้วกดรันได้ทันที
    แต่การวัดผลต้องใช้ทั้งสามไฟล์พร้อมกัน — จึงต้องมีตัวประกอบอยู่ข้างนอก
    ตัวประกอบนั้นคือไฟล์นี้ ไม่ใช่ import ที่แอบใส่กลับเข้าไปใน algorithms/

ใช้แก้อะไร
    hash_table.py ของ Kim มี TODO ค้างไว้ว่า "รอ datasets.py ของ Neo เสร็จก่อน"
    เพราะตัวเลขในไฟล์นั้นวัดบน make_sample_items() ของตัวเอง ไม่ใช่ชุด A-D
    ตัวเลขของสามหัวข้อจึงอ่านเทียบกันไม่ได้ สคริปต์นี้ปิดช่องนั้น:
    ทุกคนวัดบนชุดเดียวกันจาก make_dataset()

วิธีรัน (จากรากของ repo)
    python3 scripts/data-course-benchmark.py

ใน Colab ไม่ต้องใช้ไฟล์นี้ — วางเซลล์ตามลำดับแล้วเรียกฟังก์ชันได้ตรงๆ
    เซลล์ 1  datasets.py
    เซลล์ 2  hash_table.py
    เซลล์ 3  search.py
    เซลล์ 4  sort_group.py
"""

import sys
from pathlib import Path

# ชี้ไปที่โฟลเดอร์ algorithms/ เพื่อ import ทั้งสามไฟล์เข้ามาประกอบกัน
# การ import อยู่ "ที่นี่" ไม่ใช่ในไฟล์อัลกอริทึม จึงไม่ผิดกฎ stdlib ของไฟล์พวกนั้น
ALGORITHMS = Path(__file__).resolve().parent.parent / "backend-api" / "algorithms"
sys.path.insert(0, str(ALGORITHMS))

import datasets  # noqa: E402
import hash_table  # noqa: E402
import search  # noqa: E402


def check_datasets() -> None:
    """หน้าตาของแต่ละชุด — ต้องพิมพ์ลงรายงานเพื่อบอกว่าวัดบนอะไร"""
    print()
    print("ชุดข้อมูลที่ใช้วัด")
    print("=" * 78)
    print(f"วันอ้างอิง (TODAY) = {datasets.TODAY.isoformat()}")
    print("-" * 78)
    print(
        f"{'ชุด':>5} {'จำนวน':>9} {'ชื่อซ้ำ':>9} {'ของหมด':>9}"
        f" {'เลยกำหนด':>10} {'ชำรุด':>8}"
    )
    print("-" * 78)

    for name in ("A", "B", "C", "D"):
        items = datasets.make_dataset(name)
        info = datasets.describe(items)
        print(
            f"{name:>5} {info['จำนวน']:>9,} {info['ชื่อที่ซ้ำกัน']:>9}"
            f" {info['available = 0']:>9} {info['เลยกำหนดคืน']:>10} {info['ชำรุด']:>8}"
        )

    print("-" * 78)


def benchmark_search_all() -> None:
    """ตารางที่ 1 ของรายงาน Neo — เทียบสามวิธีค้นหาบนชุด A ถึง D"""
    rows = []

    for name in ("A", "B", "C", "D"):
        items = datasets.make_dataset(name)
        table = hash_table.build_hash_table(items)
        result = search.benchmark_search(items, table=table)
        rows.append((name, len(items), result["linear"], result["binary"], result["hash"]))

    search.print_table(
        "ตารางที่ 1 — เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) บนชุด A-D",
        rows,
    )

    # ตัวเลขสองบรรทัดนี้คือประโยคหลักของหัวข้อ Neo ในรายงาน
    small = rows[0]
    large = rows[3]
    print()
    print("สิ่งที่ตารางนี้พิสูจน์")
    print("-" * 78)
    print(
        f"    ชุด A มี {small[1]} รายการ linear เร็วกว่าหรือพอๆ กับ binary"
        f" ({small[2]:.3f} vs {small[3]:.3f} ms)"
    )
    print("    เพราะข้อมูลน้อยเกินกว่าที่การตัดครึ่งจะคุ้มค่าใช้จ่ายของการคำนวณตำแหน่ง")
    print()
    print(
        f"    ชุด D มี {large[1]:,} รายการ binary เร็วกว่า linear"
        f" {large[2] / large[3]:.1f} เท่า ({large[2]:.3f} vs {large[3]:.3f} ms)"
    )
    print(f"    และ hash เร็วกว่า linear {large[2] / large[4]:.1f} เท่า ({large[4]:.3f} ms)")
    print()
    print("    ข้อมูลโตขึ้น 100 เท่าจาก A ไป D แต่ทั้งสามวิธีโตไม่เท่ากัน")
    print("    ซึ่งคือความต่างระหว่าง O(n) · O(log n) · O(1) ที่วัดออกมาเป็นตัวเลขได้จริง")


def benchmark_counts() -> None:
    """ตารางที่ 2 — ผลการนับบนแต่ละชุด ใช้ตรวจว่าชุดข้อมูลทำหน้าที่ของมันจริง"""
    print()
    print("ตารางที่ 2 — ผลการนับบนชุด A-D")
    print("-" * 78)
    print(f"{'ชุด':>5} {'ว่างบนชั้น':>12} {'ok':>6} {'out':>6} {'broken':>8} {'เลยกำหนด':>10}")
    print("-" * 78)

    today = datasets.TODAY.isoformat()

    for name in ("A", "B", "C", "D"):
        items = datasets.make_dataset(name)
        counts = search.count_by_status(items)
        print(
            f"{name:>5} {search.count_available(items):>12,}"
            f" {counts['ok']:>6} {counts['out']:>6} {counts['broken']:>8}"
            f" {search.count_overdue(items, today):>10}"
        )

    print("-" * 78)


def check_duplicate_names() -> None:
    """ชุด B มีไว้พิสูจน์ข้อนี้ข้อเดียว — ชื่อซ้ำแล้วยังแยกออกว่าเป็นตัวไหน"""
    items = datasets.make_dataset("B")

    # หยิบชื่อแรกที่ซ้ำ
    seen = {}
    duplicate_name = None
    for item in items:
        if item["name"] in seen:
            duplicate_name = item["name"]
            break
        seen[item["name"]] = True

    found = search.search_by_name(items, duplicate_name)

    print()
    print("ชุด B — ค้นด้วยชื่อที่ซ้ำกัน")
    print("-" * 78)
    print(f'    ค้นคำว่า "{duplicate_name}" ได้ {len(found)} รายการ')
    for item in found:
        print(f"        {item['code']}  ว่าง {item['available']}/{item['total']}  {item['status']}")
    print()
    print("    ถ้า search_by_name คืนตัวแรกที่เจอแบบ linear_search ผู้ใช้จะได้ของผิดชิ้น")
    print("    แปลว่าปล่อยเช่าผิดชิ้น และอีกชิ้นจะหายไปจากระบบเงียบๆ")

    # จำนวนที่ได้มากกว่าคู่ที่ตั้งใจให้ซ้ำ เพราะค้นแบบ "มีคำนี้อยู่ข้างใน"
    # คำว่า "รุ่นที่ 1" เป็นส่วนหนึ่งของ "รุ่นที่ 11" และ "รุ่นที่ 12" ด้วย
    # พฤติกรรมนี้ถูกแล้วสำหรับช่องค้นหาในหน้าเว็บ ผู้ใช้พิมพ์ไม่ครบก็ยังเจอ
    # แต่ต้องรู้ตัวว่ามันกว้างกว่าการเทียบชื่อแบบตรงเป๊ะ — เขียนข้อนี้ลงรายงานด้วย
    exact = [item for item in items if item["name"] == duplicate_name]
    print()
    print(f"    ในจำนวนนี้ ชื่อตรงกันเป๊ะจริงๆ มี {len(exact)} รายการ")
    print('    ที่เกินมาคือชื่อที่มีคำนี้อยู่ข้างใน เช่น "รุ่นที่ 11" ก็มีคำว่า "รุ่นที่ 1" อยู่')
    print("    ค้นแบบนี้ถูกสำหรับช่องค้นหาในหน้าเว็บ เพราะผู้ใช้พิมพ์ไม่ครบก็ยังเจอ")

    # และแต่ละตัวยังหาเจอด้วย code ของตัวเองอยู่ ไม่ได้ปนกัน
    sorted_items = search.sort_by_code(items)
    for item in found:
        assert search.binary_search(sorted_items, item["code"]) == item
    print()
    print(f"    ตรวจแล้ว: ทั้ง {len(found)} ตัวยังค้นเจอด้วย code ของตัวเอง ไม่ได้ปนกัน")


def check_range_query() -> None:
    """คำถามแบบเป็นช่วง — เหตุผลว่าทำไมยังต้องมี binary search ทั้งที่ hash เร็วกว่า"""
    items = datasets.make_dataset("D")
    today = datasets.TODAY.isoformat()

    with_due = sorted(
        [item for item in items if item["due_date"] is not None],
        key=lambda item: item["due_date"],
    )

    overdue = search.find_due_before(with_due, today)
    counted = search.count_overdue(items, today)

    print()
    print("ชุด D — คำถามที่ตารางแฮชตอบไม่ได้")
    print("-" * 78)
    print(f'    "มีของอะไรบ้างที่ครบกำหนดคืนก่อน {today}"')
    print(f"    หาแบบช่วงด้วย binary search ได้ {len(overdue)} รายการ")
    print(f"    นับทีละตัวแบบ O(n) ได้ {counted} รายการ  ->  ตรงกัน")
    assert len(overdue) == counted

    print()
    print("    ตารางแฮชคำนวณตำแหน่งจาก key ที่ตรงเป๊ะเท่านั้น")
    print("    คำถามแบบ 'ก่อนวันที่ X' จึงต้องไล่ดูทั้งตาราง กลายเป็น O(n)")
    print("    ส่วน list ที่เรียงแล้ว + binary search หาขอบของช่วงได้ใน O(log n)")


def main() -> None:
    print("=" * 78)
    print("LEBIS — ผลวัดวิชา Data Structures & Algorithms")
    print("=" * 78)

    datasets.verify()
    search.verify()

    check_datasets()
    benchmark_search_all()
    benchmark_counts()
    check_duplicate_names()
    check_range_query()

    print()
    print("=" * 78)
    print("เสร็จ — ตัวเลขข้างบนคือตัวเลขที่ใส่ในรายงานได้")
    print("=" * 78)


if __name__ == "__main__":
    main()


Writing data_course_benchmark.py


In [8]:
import data_course_benchmark
data_course_benchmark.main()

LEBIS — ผลวัดวิชา Data Structures & Algorithms
ตรวจชุดข้อมูล A-D ผ่านทั้งหมด
ตรวจฟังก์ชันค้นหาและนับ ผ่านทั้งหมด

ชุดข้อมูลที่ใช้วัด
วันอ้างอิง (TODAY) = 2026-10-01
------------------------------------------------------------------------------
  ชุด     จำนวน   ชื่อซ้ำ    ของหมด   เลยกำหนด    ชำรุด
------------------------------------------------------------------------------
    A        10         0         1          3        1
    B        12         6         5          4        1
    C        10         0         5          4        1
    D     1,000         0       288        275      142
------------------------------------------------------------------------------



ตารางที่ 1 — เวลาค้นครบทุก code หนึ่งรอบ (เฉลี่ย 3 รอบ) บนชุด A-D
------------------------------------------------------------------------------
  ชุด     จำนวน    linear (ms)    binary (ms)    hash (ms)   เร็วขึ้น
------------------------------------------------------------------------------
    A        10          0.004          0.006        0.016       0.6x
    B        12          0.004          0.007        0.018       0.6x
    C        10          0.003          0.006        0.015       0.6x
    D     1,000         21.812          1.759        1.580      12.4x
------------------------------------------------------------------------------

สิ่งที่ตารางนี้พิสูจน์
------------------------------------------------------------------------------
    ชุด A มี 10 รายการ linear เร็วกว่าหรือพอๆ กับ binary (0.004 vs 0.006 ms)
    เพราะข้อมูลน้อยเกินกว่าที่การตัดครึ่งจะคุ้มค่าใช้จ่ายของการคำนวณตำแหน่ง

    ชุด D มี 1,000 รายการ binary เร็วกว่า linear 12.4 เท่า (21.812 vs 1.759 ms)
    และ h

---
## 4. Sort / Group — Merge Sort (Kul)

ตรวจความถูกต้องก่อน (ผลต้องตรงกับ `sorted()` และคง stable) แล้ววัดเวลาบนชุด A–D

In [9]:
sort_group.verify()
sort_group.report_on_datasets(datasets.make_dataset)

ตรวจฟังก์ชันเรียงลำดับและจัดกลุ่ม ผ่านทั้งหมด



เวลาเรียงทั้งชุดหนึ่งรอบ (เฉลี่ย 3 รอบ) — ชุดข้อมูลจริง A-D
------------------------------------------------------------------------------
  ชุด     จำนวน    bubble (ms)     merge (ms)   builtin (ms)   เร็วขึ้น
------------------------------------------------------------------------------
    A        10          0.011          0.013          0.001       0.9x
    B        12          0.014          0.015          0.001       0.9x
    C        10          0.008          0.013          0.001       0.6x
    D     1,000         90.516          2.619          0.124      34.6x
------------------------------------------------------------------------------


In [10]:
items = datasets.make_dataset("D")

# จัดกลุ่มรอบเดียว O(n) — ไม่ต้องวนแยกทีละหมวด
by_type = sort_group.group_by_type(items)
by_status = sort_group.group_by_status(items)
print("ตามหมวด :", {name: len(group) for name, group in by_type.items()})
print("ตามสถานะ:", {name: len(group) for name, group in by_status.items()})

ตามหมวด : {'เลนส์': 200, 'ขาตั้ง': 200, 'ไฟ': 200, 'ไมโครโฟน': 200, 'กล้อง': 200}
ตามสถานะ: {'ok': 616, 'out': 242, 'broken': 142}


---
## 5. สามอัลกอริทึมทำงานต่อกัน

คำถามที่ร้านถามทุกเช้า: **"ใบไหนเลยกำหนดคืนแล้วบ้าง แล้วของชิ้นนั้นคืออะไร"**

1. **merge sort (Kul)** เรียงใบยืมตามวันครบกำหนด — O(n log n)
2. **binary search (Neo)** ตัดเอาทุกใบที่ครบกำหนดก่อนวันนี้ — O(log n + k)
3. **hash table (Kim)** เปิดดูรายละเอียดของแต่ละชิ้นด้วยรหัส — O(1) ต่อชิ้น

แต่ละขั้นส่ง list ธรรมดาต่อให้ขั้นถัดไป ถ้าใครเปลี่ยนรูปแบบข้อมูล เซลล์นี้จะล้มทันที

In [11]:
TODAY = "2026-10-01"
items = datasets.make_dataset("D")

by_due = sort_group.sort_by_due_date(items)          # Kul — merge sort
overdue = search.find_due_before(by_due, TODAY)       # Neo — binary search
table = hash_table.build_hash_table(items)            # Kim — hash table

# เทียบกับการไล่ทีละตัว ต้องได้ชุดเดียวกันเป๊ะ
expected = sorted(x["code"] for x in items if x["due_date"] is not None and x["due_date"] < TODAY)
assert sorted(x["code"] for x in overdue) == expected

print(f"ใบยืมทั้งหมด {len(by_due)} ใบ · เลยกำหนดก่อน {TODAY}: {len(overdue)} ใบ (ตรงกับการไล่ทีละตัว)")
for entry in overdue[:5]:
    item = table.get(entry["code"])
    print(f"  {item['code']:<10} {item['name']:<22} ผู้ยืม {item['borrower']:<12} ครบกำหนด {item['due_date']}")

ใบยืมทั้งหมด 685 ใบ · เลยกำหนดก่อน 2026-10-01: 275 ใบ (ตรงกับการไล่ทีละตัว)
  MIC-0004   ไมโครโฟน รุ่นที่ 4     ผู้ยืม กมลชนก       ครบกำหนด 2026-09-21
  LEN-0081   เลนส์ รุ่นที่ 81       ผู้ยืม ธนกร         ครบกำหนด 2026-09-21
  MIC-0104   ไมโครโฟน รุ่นที่ 104   ผู้ยืม ปวีณา        ครบกำหนด 2026-09-21
  CAM-0125   กล้อง รุ่นที่ 125      ผู้ยืม อานนท์       ครบกำหนด 2026-09-21
  TRI-0142   ขาตั้ง รุ่นที่ 142     ผู้ยืม พิมพ์ชนก     ครบกำหนด 2026-09-21
